# Random Forest - Cross Validation (UCI-THYROID-DXBIN)

Notebook para executar Random Forest com StratifiedKFold sobre o dataset UCI-THYROID-DXBIN.
- Selecione qual dataset normalizado usar alterando a variável `DATASET_PATH` no bloco de configuração.
- Os parâmetros do modelo estão em `rf_params`.
- Outputs: `model_reports/random_forest/cv/` contém `csv/`, `images/`, `pdf/`.
- O modelo será salvo em `databases/UCI-THYROID-DXBIN/common/models/random_forest/v1/rf_model_cf{N_SPLITS}.pkl`.

In [1]:
# Configuração
import os
from pathlib import Path
import warnings
warnings.filterwarnings('ignore')

BASE = Path('../../')  # ajustável se necessário
# Escolha um dos 3 datasets normalizados gerados pelo EDA: 'Standard', 'Robust', 'MinMax'
DATASET_NAME = 'uci_thyroid_dxbin_processed.csv'  # alterar para uci_thyroid_dxbin_processed.csv
DATASET_PATH = Path('../data/processed') / DATASET_NAME
# Se quiser forçar explicitamente qual coluna é o target (no bruto ou no normalizado), defina aqui
# Exemplo: TARGET_COLUMN = 'diagnosis' ou TARGET_COLUMN = 'y'
TARGET_COLUMN = 'y' # ajustar se necessário

# Output paths
REPORTS = Path('../model_reports/random_forest/cv')
CSV_OUT = REPORTS / 'csv'
IMAGES_OUT = REPORTS / 'images'
PDF_OUT = REPORTS / 'pdf'
MODEL_DIR = Path('../common/models/random_forest/v1')
# Basico outputs (single train/test) - sibling folder to 'cv'
BASICO_DIR = REPORTS.parent / 'basico'
BASICO_CSV = BASICO_DIR / 'csv'
BASICO_IMAGES = BASICO_DIR / 'images'

for d in [REPORTS, CSV_OUT, IMAGES_OUT, PDF_OUT, MODEL_DIR, BASICO_DIR, BASICO_CSV, BASICO_IMAGES]:
    d.mkdir(parents=True, exist_ok=True)

# Random Forest params (variável conforme solicitado)
rf_params = {
    "n_estimators": 650,
    "max_depth": 16,
    "min_samples_split": 8,
    "min_samples_leaf": 15,
    "max_features": 0.3,
    "bootstrap": True,
    "class_weight": "balanced_subsample",
    "criterion": "entropy",
    'n_jobs': 8
}

# CV and threshold config
N_SPLITS = 10
THRESHOLD = 0.5
# Colunas identificadoras que NÃO devem entrar como features no modelo, mas devem ser mantidas nos arquivos de saída
EXCLUDE_COLUMNS = ['id', 'ID', 'patient_id', 'pid', '_target', 'target','target_','y']
VARIABLE_DISCRETIZE = ['sex', 'on_thyroxine', 'query_on_thyroxine', 'on_antithyroid_meds', 'sick', 'pregnant', 'thyroid_surgery', 'I131_treatment', 'query_hypothyroid', 'query_hyperthyroid', 'lithium', 'goitre', 'tumor', 'hypopituitary', 'psych', 'TSH_measured', 'TSH', 'T3_measured', 'T3', 'TT4_measured', 'TT4', 'T4U_measured', 'T4U', 'FTI_measured', 'FTI', 'TBG_measured']
VARIABLE_DISCRETIZE_NUMBER =  { 'other':0, 'SVI':1, 'SVHC':2, 'STMW':3, 'SVHD':4, 'WEST':5}

# Pareto configuration: threshold (fraction) and selected-features placeholder (set to None to be updated after CV)
PARETO_THRESHOLD = 0.9  # fraction, e.g. 0.8 == 80%
PARETO_SELECTED_FEATURES = None

# Variance percent-difference threshold (fraction). Example: 0.1 == 10%% difference.
VAR_DIFF_PCT_THRESHOLD = 0.01
TARGET_COLUMN = 'y'  # ajustar se necessário
TEST_SIZE = 0.3
RT_RANDOM_STATE = 42


In [2]:
# Imports principais
import time
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns
from sklearn.ensemble import RandomForestClassifier
from sklearn.model_selection import StratifiedKFold
from sklearn.metrics import (accuracy_score, f1_score, roc_auc_score, precision_score, recall_score,
                             balanced_accuracy_score, confusion_matrix, roc_curve, log_loss, auc)
from matplotlib.backends.backend_pdf import PdfPages
from tqdm.auto import tqdm
import joblib

sns.set(style='whitegrid')

In [3]:
# Load dataset (path definido na configuração)
import shutil
if not DATASET_PATH.exists():
    raise FileNotFoundError(f'Arquivo de dataset não encontrado: {DATASET_PATH.resolve()}')

# Preserve a raw copy of the original CSV (unaltered) in data/processed with suffix _raw
RAW_COPY_PATH = Path('../data/processed') / f"{DATASET_NAME.replace('.csv','')}_raw.csv"
try:
    shutil.copy2(DATASET_PATH, RAW_COPY_PATH)
    print(f'Raw dataset copiado para: {RAW_COPY_PATH}')
except Exception as e_copy:
    print('Não foi possível copiar o CSV original para processed:', e_copy)

# Leia o dataset (continuamos a usar df para processamento posterior)
df = pd.read_csv(DATASET_PATH)
# Função utilitária para encontrar coluna aproximada (case/strip tolerant)
def find_column(df, name):
    # tenta match exato
    if name in df.columns:
        return name
    # tenta match case-insensitive
    low = {c.lower().strip(): c for c in df.columns}
    if name.lower().strip() in low:
        return low[name.lower().strip()]
    # tenta encontrar coluna que contenha o nome
    for c in df.columns:
        if name.lower().strip() in c.lower():
            return c
    return None

# Determina a coluna alvo: usa TARGET_COLUMN quando fornecida, senão infere
target_col = None
if TARGET_COLUMN:
    candidate = find_column(df, TARGET_COLUMN)
    if candidate is None:
        raise ValueError(f"TARGET_COLUMN='{TARGET_COLUMN}' não encontrada. Colunas: {list(df.columns)}")
    target_col = candidate
else:
    # heurística leve: nomes comuns
    for cand in ['y', 'diagnosis', '_target', 'target', 'label', 'class', 'diagnose']:
        candidate = find_column(df, cand)
        if candidate:
            target_col = candidate
            break
    # procura colunas binárias
    if target_col is None:
        for c in df.columns:
            if c.lower().strip() in ('id', 'patient_id', 'pid'):
                continue
            try:
                nunq = df[c].nunique(dropna=True)
            except Exception:
                nunq = 999
            if nunq == 2:
                target_col = c
                break
    # fallback para última coluna com poucos valores distintos
    if target_col is None:
        lastc = df.columns[-1]
        if df[lastc].nunique(dropna=True) <= 10:
            target_col = lastc
    if target_col is None:
        raise ValueError(f"Não foi possível inferir a coluna alvo. Defina TARGET_COLUMN ou verifique as colunas: {list(df.columns)}")

# Renomeia para 'y' e garante coluna disponível antes de acessar
#if target_col != 'y':
#    df = df.rename(columns={target_col: 'y'})
if 'y' not in df.columns:
    raise KeyError('Coluna y não encontrada após renomeação')

# Se y não for numérico, discretiza (M/B -> 1/0 ou factorize para múltiplas classes)
#if not np.issubdtype(df['y'].dtype, np.number):
#    unique_vals = list(df['y'].dropna().unique())
#    print(unique_vals)
#    low = [str(u).lower() for u in unique_vals]
#    if set(low) <= set(['T', 'F']):
#        mapping = {v: (1 if str(v)=='T' else 0) for v in unique_vals}
#        df['y'] = df['y'].map(mapping)
#    else:
#        # factorize para inteiros 0..k-1
#        df['y'], uniques = pd.factorize(df['y'])
#    # salva versão discretizada para inspeção
#    print(f'CSV_OUT / database_used_{DATASET_NAME}_with_y_numeric.csv')
#    df.to_csv(CSV_OUT / f'database_used_{DATASET_NAME}_with_y_numeric.csv', index=False)

# Substitui inf valores e dropna (pode ajustar imputação se preferir)
#df.replace([np.inf, -np.inf], np.nan, inplace=True)
#df = df.dropna()

# reorganiza para garantir que y seja a última coluna
#cols = [c for c in df.columns if c != 'y'] + ['y']
#df = df[cols]

# salva uma cópia de entrada no csv de reports (final)
df.to_csv(CSV_OUT / f'database_used_{DATASET_NAME}', index=False)

print('Dataset shape:', df.shape)

Raw dataset copiado para: ..\data\processed\uci_thyroid_dxbin_processed_raw.csv
Dataset shape: (8025, 32)


In [ ]:
# # Preparação dos dados X, y
# y = df['y']
# # Detecta colunas identificadoras a partir de EXCLUDE_COLUMNS (case-insensitive)
# exclude_lower = {c.lower() for c in EXCLUDE_COLUMNS}
# ID_COLS = [c for c in df.columns if c.lower() in exclude_lower]
# # garante que id columns não contenham a coluna alvo y
# ID_COLS = [c for c in ID_COLS if c != 'y']
# # FEATURES são todas as colunas exceto ids e y
# FEATURES = [c for c in df.columns if c not in ID_COLS + ['y']]
# # X_model será usado para treinar (sem ids)
# X_model = df[FEATURES].copy()
# X_model.replace([np.inf, -np.inf], np.nan, inplace=True)
# # preenche valores faltantes simples (pode ser alterado)
# X_model = X_model.fillna(X_model.median())

# kf = StratifiedKFold(n_splits=N_SPLITS, shuffle=True, random_state=42)

# # estruturas de resultado
# resultados = []
# # listas para métricas agregadas por fold (treino/teste)
# aurocs_train = []
# aurocs_test = []
# fprs_train = []
# tprs_train = []
# fprs_test = []
# tprs_test = []
# importancias_lista = []
# X_train_parts = []
# X_test_parts = []
# y_train_parts = []
# y_test_parts = []
# y_train_pred_parts = []
# y_test_pred_parts = []

# start_total = time.time()

# with PdfPages(PDF_OUT / f'RF_CV_{N_SPLITS}.pdf') as pdf:
#     with tqdm(total=N_SPLITS, desc="🔄 Folds K-Fold") as pbar:
#         for fold, (train_idx, test_idx) in enumerate(kf.split(X_model, y), 1):
#             t0 = time.time()
#             # Para manter ids nos arquivos de saída, indexamos sobre o DataFrame original
#             X_train_full = df.iloc[train_idx][ID_COLS + FEATURES] if len(ID_COLS)>0 else df.iloc[train_idx][FEATURES]
#             X_test_full = df.iloc[test_idx][ID_COLS + FEATURES] if len(ID_COLS)>0 else df.iloc[test_idx][FEATURES]
#             # Usamos apenas as FEATURES (sem ids) para treinar o modelo
#             X_train = X_train_full[FEATURES].copy()
#             X_test = X_test_full[FEATURES].copy()
#             y_train, y_test = y.iloc[train_idx], y.iloc[test_idx]

#             model = RandomForestClassifier(**rf_params)
#             model.fit(X_train, y_train)

#             # Previsões de probabilidade
#             y_proba_test_all = model.predict_proba(X_test)
#             y_proba_test_C1 = y_proba_test_all[:, 1]
#             y_proba_test_C1_bin = (y_proba_test_C1 >= THRESHOLD).astype(int)

#             y_proba_train_all = model.predict_proba(X_train)
#             y_proba_train_C1 = y_proba_train_all[:, 1]
#             y_proba_train_C1_bin = (y_proba_train_C1 >= THRESHOLD).astype(int)

#             # guarda partes completas (incluem ids) para concat posterior e rastreabilidade
#             # mantemos o índice original para fácil merge posterior
#             X_test_parts.append(X_test_full.assign(fold=fold).reset_index())
#             y_test_parts.append(pd.Series(y_test, name='y_test').to_frame().assign(fold=fold).reset_index())
#             # ensure probability dataframe uses the original row indices so reset_index() produces the original index column
#             y_test_pred_parts.append(pd.DataFrame(y_proba_test_all, columns=['prob_0', 'prob_1'], index=X_test_full.index).assign(fold=fold).reset_index())

#             X_train_parts.append(X_train_full.assign(fold=fold).reset_index())
#             y_train_parts.append(pd.Series(y_train, name='y_train').to_frame().assign(fold=fold).reset_index())
#             # ensure probability dataframe uses the original row indices so reset_index() produces the original index column
#             y_train_pred_parts.append(pd.DataFrame(y_proba_train_all, columns=['prob_0', 'prob_1'], index=X_train_full.index).assign(fold=fold).reset_index())

#             # Métricas Treino
#             cm_train = confusion_matrix(y_train, y_proba_train_C1_bin)
#             tn_tr, fp_tr, fn_tr, tp_tr = cm_train.ravel()
#             total_cm_treino = cm_train.sum()
#             fpr, tpr, _ = roc_curve(y_train, y_proba_train_C1)
#             fprs_train.append(fpr)
#             tprs_train.append(tpr)
#             aurocs_train.append(roc_auc_score(y_train, y_proba_train_C1))

#             MASK_0_TRAIN = (y_train == 0)
#             MASK_1_TRAIN = (y_train == 1)
#             logloss_train_0 = log_loss(y_train[MASK_0_TRAIN], y_proba_train_C1[MASK_0_TRAIN], labels=[0,1]) if MASK_0_TRAIN.sum()>0 else np.nan
#             logloss_train_1 = log_loss(y_train[MASK_1_TRAIN], y_proba_train_C1[MASK_1_TRAIN], labels=[0,1]) if MASK_1_TRAIN.sum()>0 else np.nan
#             cross_prop_train = pd.Series(y_train).value_counts(normalize=True) * 100

#             # Métricas Teste
#             cm_test = confusion_matrix(y_test, y_proba_test_C1_bin)
#             tn_ts, fp_ts, fn_ts, tp_ts = cm_test.ravel()
#             total_cm_teste = cm_test.sum()
#             fpr, tpr, _ = roc_curve(y_test, y_proba_test_C1)
#             fprs_test.append(fpr)
#             tprs_test.append(tpr)
#             aurocs_test.append(roc_auc_score(y_test, y_proba_test_C1))
#             MASK_0_TEST = (y_test == 0)
#             MASK_1_TEST = (y_test == 1)
#             logloss_test_0 = log_loss(y_test[MASK_0_TEST], y_proba_test_C1[MASK_0_TEST], labels=[0,1]) if MASK_0_TEST.sum()>0 else np.nan
#             logloss_test_1 = log_loss(y_test[MASK_1_TEST], y_proba_test_C1[MASK_1_TEST], labels=[0,1]) if MASK_1_TEST.sum()>0 else np.nan
#             cross_prop_test = pd.Series(y_test).value_counts(normalize=True) * 100

#             # Armazena resultados (Treino/Teste)
#             resultados.append({
#                 'Conjunto': 'Treino',
#                 'Fold': fold,
#                 'Acurácia': accuracy_score(y_train, y_proba_train_C1_bin),
#                 'Cross-Entropy C0': logloss_train_0,
#                 'Proporção C0': cross_prop_train.iloc[0] if 0 in cross_prop_train.index else np.nan,
#                 'Cross-Entropy C1': logloss_train_1,
#                 'Proporção C1': cross_prop_train.iloc[1] if 1 in cross_prop_train.index else np.nan,
#                 'F1 Score': f1_score(y_train, y_proba_train_C1_bin),
#                 'Balanced Accuracy': balanced_accuracy_score(y_train, y_proba_train_C1_bin),
#                 'Precision': precision_score(y_train, y_proba_train_C1_bin),
#                 'Recall': recall_score(y_train, y_proba_train_C1_bin),
#                 'AUROC': aurocs_train,
#                 'TP': tp_tr, 'FP': fp_tr, 'TN': tn_tr, 'FN': fn_tr,
#                 'TP(%)': round(tp_tr / total_cm_treino * 100, 2),
#                 'FP(%)': round(fp_tr / total_cm_treino * 100, 2),
#                 'TN(%)': round(tn_tr / total_cm_treino * 100, 2),
#                 'FN(%)': round(fn_tr / total_cm_treino * 100, 2),
#                 'Tempo (s)': round(time.time() - t0, 2)
#             })

#             resultados.append({
#                 'Conjunto': 'Teste',
#                 'Fold': fold,
#                 'Acurácia': accuracy_score(y_test, y_proba_test_C1_bin),
#                 'Cross-Entropy C0': logloss_test_0,
#                 'Proporção C0': cross_prop_test.iloc[0] if 0 in cross_prop_test.index else np.nan,
#                 'Cross-Entropy C1': logloss_test_1,
#                 'Proporção C1': cross_prop_test.iloc[1] if 1 in cross_prop_test.index else np.nan,
#                 'F1 Score': f1_score(y_test, y_proba_test_C1_bin),
#                 'Balanced Accuracy': balanced_accuracy_score(y_test, y_proba_test_C1_bin),
#                 'Precision': precision_score(y_test, y_proba_test_C1_bin),
#                 'Recall': recall_score(y_test, y_proba_test_C1_bin),
#                 'AUROC': aurocs_test,
#                 'TP': tp_ts, 'FP': fp_ts, 'TN': tn_ts, 'FN': fn_ts,
#                 'TP(%)': round(tp_ts / total_cm_teste * 100, 2),
#                 'FP(%)': round(fp_ts / total_cm_teste * 100, 2),
#                 'TN(%)': round(tn_ts / total_cm_teste * 100, 2),
#                 'FN(%)': round(fn_ts / total_cm_teste * 100, 2),
#                 'Tempo (s)': round(time.time() - t0, 2)
#             })

#             # Importância das features (percentual)
#             importancias = model.feature_importances_
#             nomes_colunas = FEATURES
#             linha_importancia = {'Conjunto': 'Teste', 'Fold': fold}
#             for nome, valor in zip(nomes_colunas, importancias):
#                 linha_importancia[nome] = f"{valor * 100:.2f}%"
#             importancias_lista.append(linha_importancia)

#             # salva modelo por fold (opcional)
#             model_path = MODEL_DIR / f'rf_model_cf{N_SPLITS}.pkl'
#             joblib.dump(model, model_path)

#             pbar.update(1)

#     # fim do loop folds

# # Concatena e salva os resultados
# df_resultados = pd.DataFrame(resultados)
# # adiciona média das métricas
# mean_row = df_resultados.select_dtypes(include=[np.number]).mean()
# mean_row['Conjunto'] = 'Média'
# mean_row['Fold'] = 'Média'
# df_resultados = pd.concat([df_resultados, pd.DataFrame([mean_row])], ignore_index=True, sort=False)

# # salva csvs principais
# df_resultados.to_csv(CSV_OUT / f'rf_cv_results_{N_SPLITS}.csv', index=False)
# pd.DataFrame(importancias_lista).to_csv(CSV_OUT / f'rf_feature_importances_{N_SPLITS}.csv', index=False)

# # Concatena blocos para gerar dataframes completos de treino/teste com probabilidades
# if len(X_train_parts)>0:
#     X_train_global = pd.concat(X_train_parts, ignore_index=True)
#     y_train_global = pd.concat(y_train_parts, ignore_index=True)
#     y_train_pred_global = pd.concat(y_train_pred_parts, ignore_index=True)
# else:
#     X_train_global = pd.DataFrame(); y_train_global = pd.DataFrame(); y_train_pred_global = pd.DataFrame()

# if len(X_test_parts)>0:
#     X_test_global = pd.concat(X_test_parts, ignore_index=True)
#     y_test_global = pd.concat(y_test_parts, ignore_index=True)
#     y_test_pred_global = pd.concat(y_test_pred_parts, ignore_index=True)
# else:
#     X_test_global = pd.DataFrame(); y_test_global = pd.DataFrame(); y_test_pred_global = pd.DataFrame()

# # Ajusta nomes: os objetos concat possuem coluna 'index' que era o índice original; renomeamos para 'orig_index' para merge limpo
# for df_block in [X_train_global, X_test_global, y_train_global, y_test_global, y_train_pred_global, y_test_pred_global]:
#     if 'index' in df_block.columns:
#         df_block.rename(columns={'index': 'orig_index'}, inplace=True)

# # Mescla para criar arquivos finais com probas e fold
# if not X_train_global.empty and not y_train_global.empty:
#     # merge features + true y + predicted probabilities (all have orig_index and fold)
#     train_df = X_train_global.merge(y_train_global, on=['orig_index', 'fold'], how='left')
#     if not y_train_pred_global.empty:
#         train_df = train_df.merge(y_train_pred_global, on=['orig_index', 'fold'], how='left')
# else:
#     train_df = pd.DataFrame()

# if not X_test_global.empty and not y_test_global.empty:
#     test_df = X_test_global.merge(y_test_global, on=['orig_index', 'fold'], how='left')
#     if not y_test_pred_global.empty:
#         test_df = test_df.merge(y_test_pred_global, on=['orig_index', 'fold'], how='left')
# else:
#     test_df = pd.DataFrame()

# # Salva os datasets com probabilidades (prob_0, prob_1) e já binarizados por threshold
# if not test_df.empty:
#     test_df['y_proba'] = test_df['prob_1']
#     test_df['y_pred'] = (test_df['y_proba'] >= THRESHOLD).astype(int)
#     test_df.to_csv(CSV_OUT / f'rf_test_with_proba_{N_SPLITS}.csv', index=False)
# if not train_df.empty:
#     train_df['y_proba'] = train_df['prob_1']
#     train_df['y_pred'] = (train_df['y_proba'] >= THRESHOLD).astype(int)
#     train_df.to_csv(CSV_OUT / f'rf_train_with_proba_{N_SPLITS}.csv', index=False)

# # Salva um dataset unificado em process (com y_proba e y_pred sobre todo o conjunto de teste concatenado)
# if not test_df.empty:
#     test_df.to_csv(Path('../data/processed') / f'uci_thyroid_dxbin_rf_test_with_proba_{N_SPLITS}.csv', index=False)

# # create a full-augmented dataset for the original input dataset with y, y_pred, y_proba
# try:
#     # we will attempt to create 'df_augmented' by merging test_df (predictions) back into the original df
#     if not test_df.empty:
#         # test_df contains orig_index referencing original df index; ensure orig_index present
#         df_aug = df.reset_index().rename(columns={'index': 'orig_index'})
#         merge_cols = [c for c in test_df.columns if c not in df_aug.columns]
#         # select only prediction cols to append
#         preds = test_df[['orig_index', 'y_pred', 'y_proba']]
#         df_augmented = pd.merge(df_aug, preds, how='left', on='orig_index')
#         # write to cv CSV folder
#         df_augmented.to_csv(CSV_OUT / f'database_used_{DATASET_NAME}_with_preds_{N_SPLITS}.csv', index=False)
#     else:
#         print('test_df vazio: não foi possível gerar dataset augmentado com previsões')
# except Exception as e_aug:
#     print('Erro ao gerar dataset augmentado com previsões:', e_aug)

# print('Resultados salvos em:', CSV_OUT)


In [4]:
# ================== OPTUNA (TPE + PRUNER) — nested 8/1/1 com 10 folds ==================
import optuna
from optuna.samplers import TPESampler
from optuna.pruners import MedianPruner
from optuna.exceptions import TrialPruned

from pathlib import Path
import numpy as np
import pandas as pd

from sklearn.model_selection import StratifiedKFold
from sklearn.metrics import (roc_auc_score, balanced_accuracy_score,
                             accuracy_score, f1_score, precision_score, recall_score, confusion_matrix)
from sklearn.ensemble import RandomForestClassifier
from sklearn.preprocessing import OrdinalEncoder

# --------- Saídas ---------
if 'CSV_OUT' not in globals():
    CSV_OUT = Path('./cv_out')
CSV_OUT.mkdir(parents=True, exist_ok=True)

# --------- Helpers já existentes (fallbacks) ----------
if '_apply_fixed_mapping_referral' not in globals():
    def _apply_fixed_mapping_referral(df_in):
        if 'referral_source' in df_in.columns and 'VARIABLE_DISCRETIZE_NUMBER' in globals():
            mapped = df_in['referral_source'].map(VARIABLE_DISCRETIZE_NUMBER)
            df_in = df_in.copy()
            df_in['referral_source'] = mapped.fillna(-1).astype('int64')
        return df_in

if '_split_cat_num_columns' not in globals():
    def _split_cat_num_columns(X, variable_discretize):
        non_numeric = [c for c in X.columns if not pd.api.types.is_numeric_dtype(X[c])]
        extra = [c for c in variable_discretize if (c in X.columns and not pd.api.types.is_numeric_dtype(X[c]))] \
                if 'VARIABLE_DISCRETIZE' in globals() else []
        cat_cols = sorted(list(dict.fromkeys(non_numeric + extra)))
        num_cols = [c for c in X.columns if c not in cat_cols]
        return cat_cols, num_cols

if '_fit_transform_fold' not in globals():
    def _fit_transform_fold(Xtr, Xte, variable_discretize):
        Xtr_m = _apply_fixed_mapping_referral(Xtr)
        Xte_m = _apply_fixed_mapping_referral(Xte)
        cat_cols, num_cols = _split_cat_num_columns(Xtr_m, variable_discretize if 'VARIABLE_DISCRETIZE' in globals() else [])
        if len(cat_cols) > 0:
            enc = OrdinalEncoder(handle_unknown='use_encoded_value', unknown_value=-1, dtype=np.int64)
            Xtr_cat = enc.fit_transform(Xtr_m[cat_cols])
            Xte_cat = enc.transform(Xte_m[cat_cols])
            Xtr_enc = pd.concat(
                [
                    Xtr_m[num_cols].reset_index(drop=True),
                    pd.DataFrame(Xtr_cat, columns=cat_cols, index=Xtr_m.index).reset_index(drop=True)
                ],
                axis=1
            )
            Xte_enc = pd.concat(
                [
                    Xte_m[num_cols].reset_index(drop=True),
                    pd.DataFrame(Xte_cat, columns=cat_cols, index=Xte_m.index).reset_index(drop=True)
                ],
                axis=1
            )
        else:
            Xtr_enc, Xte_enc = Xtr_m.copy(), Xte_m.copy()
        # sanity
        if Xtr_enc.isna().any().any():
            raise ValueError(f"Há NaN em X_train codificado: {Xtr_enc.columns[Xtr_enc.isna().any()].tolist()}")
        if Xte_enc.isna().any().any():
            raise ValueError(f"Há NaN em X_test codificado: {Xte_enc.columns[Xte_enc.isna().any()].tolist()}")
        return Xtr_enc, Xte_enc

# --------- Configurações ----------
SEED          = 42
N_TRIALS      = 3                                   # ajustável
N_OUTER_FOLDS = 10                                   # 10 folds totais
N_INNER_FOLDS = 8                                    # 8 folds internos (tuning)

# FEATURES (garantia)
if 'FEATURES' not in globals():
    exclude_lower = {c.lower() for c in EXCLUDE_COLUMNS} if 'EXCLUDE_COLUMNS' in globals() else set()
    ID_COLS = [c for c in df.columns if c.lower() in exclude_lower]
    ID_COLS = [c for c in ID_COLS if c != 'y']
    FEATURES = [c for c in df.columns if c not in ID_COLS + ['y']]

y = df['y']
classes_ordenadas = np.sort(pd.unique(y))

# ===============================
# Construção dos 10 folds estratificados
# ===============================
outer_kfold = StratifiedKFold(n_splits=N_OUTER_FOLDS, shuffle=True, random_state=SEED)

fold_indices = []  # lista com test_idx de cada fold (10 elementos)
for _, test_idx in outer_kfold.split(df[FEATURES], y):
    fold_indices.append(test_idx)

if len(fold_indices) != 10:
    raise RuntimeError(f"Esperados 10 folds, obtido {len(fold_indices)}")

# Folds 1–8 => tuning interno
train_cv_idx = np.concatenate(fold_indices[0:8])
# Fold 9 => validação externa
val_ext_idx  = fold_indices[8]
# Fold 10 => hold-out de teste final
test_holdout_idx = fold_indices[9]

df_tr = df.iloc[train_cv_idx].reset_index(drop=True)      # 8 folds (tuning)
df_va = df.iloc[val_ext_idx].reset_index(drop=True)       # fold 9 (val externa)
df_ts = df.iloc[test_holdout_idx].reset_index(drop=True)  # fold 10 (teste final)

print(f"📐 Esquema nested 10 folds:")
print(f" - Folds 1–8  → tuning interno (Optuna, CV {N_INNER_FOLDS}-fold)")
print(f" - Fold 9     → validação externa (fold=9)")
print(f" - Fold 10    → teste hold-out final (fold=10)")

# =========================
# Objetivo (CV interno em df_tr = 8 folds)
# =========================
def _suggest_params(trial: optuna.Trial):
    params = dict(
        n_estimators      = trial.suggest_int('n_estimators', 400, 2000, step=50),
        max_depth         = trial.suggest_categorical('max_depth', [None] + list(range(4, 41))),
        min_samples_split = trial.suggest_int('min_samples_split', 2, 30),
        min_samples_leaf  = trial.suggest_int('min_samples_leaf', 1, 30),
        bootstrap         = trial.suggest_categorical('bootstrap', [True, False]),
        class_weight      = trial.suggest_categorical('class_weight', [None, 'balanced', 'balanced_subsample']),
        n_jobs            = -1,
        random_state      = SEED
    )
    mf_mode = trial.suggest_categorical('max_features_mode', ['sqrt', 'log2', 'fraction'])
    if mf_mode == 'fraction':
        params['max_features'] = trial.suggest_float('max_features', 0.2, 0.9, step=0.05)
    else:
        params['max_features'] = mf_mode
    if params['bootstrap']:
        params['max_samples'] = trial.suggest_float('max_samples', 0.5, 1.0)
    return params

def _make_objective(features):
    # CV interno só nos 8 folds de df_tr
    kf = StratifiedKFold(n_splits=N_INNER_FOLDS, shuffle=True, random_state=SEED)
    X80 = df_tr[features]; y80 = df_tr['y']

    def objective(trial: optuna.Trial) -> float:
        params = _suggest_params(trial)
        fold_scores = []
        for fold, (tr_idx, te_idx) in enumerate(kf.split(X80, y80), 1):
            Xtr_raw = X80.iloc[tr_idx].copy()
            Xte_raw = X80.iloc[te_idx].copy()
            y_tr, y_te = y80.iloc[tr_idx], y80.iloc[te_idx]

            X_tr_enc, X_te_enc = _fit_transform_fold(
                Xtr_raw,
                Xte_raw,
                VARIABLE_DISCRETIZE if 'VARIABLE_DISCRETIZE' in globals() else []
            )

            model = RandomForestClassifier(**params)
            model.fit(X_tr_enc, y_tr)

            try:
                if y.nunique() == 2:
                    proba = model.predict_proba(X_te_enc)[:, list(model.classes_).index(1)]
                    score = roc_auc_score(y_te, proba)
                else:
                    proba = model.predict_proba(X_te_enc)
                    score = roc_auc_score(y_te, proba, multi_class='ovr', average='macro')
            except Exception:
                pred = model.predict(X_te_enc)
                score = balanced_accuracy_score(y_te, pred)

            fold_scores.append(score)
            trial.report(score, step=fold)
            if trial.should_prune():
                raise TrialPruned()
        return float(np.mean(fold_scores))

    return objective

# =========================
# Execução do estudo (tuning nos folds 1–8)
# =========================
study = optuna.create_study(
    direction='maximize',
    sampler=TPESampler(seed=SEED),
    pruner=MedianPruner(n_startup_trials=20, n_warmup_steps=3),
    study_name='rf_tuning_nested_811'
)
print(f"🚀 Optuna nested (8 folds internos + 1 val + 1 teste): {N_TRIALS} trials × {N_INNER_FOLDS} folds internos")
study.optimize(_make_objective(FEATURES), n_trials=N_TRIALS, show_progress_bar=True)

print("✅ Melhor valor (média AUC CV-8-fold em folds 1–8):", study.best_value)
print("✅ Melhor conjunto de hiperparâmetros (raw):")
print(study.best_params)

# --------- Normalização das chaves auxiliares -> rf_params final ----------
_best = study.best_params.copy()
mf_mode = _best.pop('max_features_mode', None)
if mf_mode != 'fraction':
    _best['max_features'] = mf_mode
rf_params = dict(n_jobs=-1, random_state=SEED, **_best)
print("\n📦 rf_params para usar no pipeline:")
print(rf_params)

# =========================
# Helper: monta linha de métricas (formato dos CSVs)
# =========================
def _compute_metrics_row(etapa, fold, y_true, y_pred, proba_mat):
    """
    etapa: 'inner_cv', 'validacao', 'teste'
    fold : 1..8 (inner), 9 (val), 10 (teste)
    y_true, y_pred: arrays 1D com classes {0,1,2}
    proba_mat: matriz (n_amostras, 3) com probs para classes 0,1,2
    """
    y_true = np.asarray(y_true)
    y_pred = np.asarray(y_pred)
    proba_mat = np.asarray(proba_mat)
    total = len(y_true)

    classes = np.array([0, 1, 2], dtype=int)

    # ---------- Cross-Entropy por classe + proporção ----------
    eps = 1e-15
    ce_per_class = {}
    prop_per_class = {}

    for k in classes:
        mask_k = (y_true == k)
        if mask_k.any():
            p_k = np.clip(proba_mat[mask_k, k], eps, 1.0 - eps)
            ce_per_class[k] = float(-np.mean(np.log(p_k)))
            prop_per_class[k] = float(mask_k.mean())
        else:
            ce_per_class[k] = np.nan
            prop_per_class[k] = 0.0

    # ---------- Matriz de confusão 3x3 ----------
    cm = confusion_matrix(y_true, y_pred, labels=classes)

    TP = {}
    FP = {}
    FN = {}
    TN = {}
    Acc_class = {}
    TP_pct = {}
    FP_pct = {}
    TN_pct = {}
    FN_pct = {}

    for idx, k in enumerate(classes):
        tp = cm[idx, idx]
        fp = cm[:, idx].sum() - tp
        fn = cm[idx, :].sum() - tp
        tn = cm.sum() - (tp + fp + fn)

        TP[k] = int(tp)
        FP[k] = int(fp)
        FN[k] = int(fn)
        TN[k] = int(tn)

        Acc_class[k] = (tp + tn) / total if total > 0 else np.nan

        TP_pct[k] = tp / total if total > 0 else np.nan
        FP_pct[k] = fp / total if total > 0 else np.nan
        TN_pct[k] = tn / total if total > 0 else np.nan
        FN_pct[k] = fn / total if total > 0 else np.nan

    # ---------- Métricas globais ----------
    acc_global = accuracy_score(y_true, y_pred)
    bal_acc    = balanced_accuracy_score(y_true, y_pred)
    f1_macro   = f1_score(y_true, y_pred, average='macro')
    f1_micro   = f1_score(y_true, y_pred, average='micro')
    f1_weight  = f1_score(y_true, y_pred, average='weighted')
    prec_macro = precision_score(y_true, y_pred, average='macro')
    rec_macro  = recall_score(y_true, y_pred, average='macro')

    return {
        'etapa': etapa,
        'fold': fold,

        # ----- globais -----
        'Acurácia': acc_global,
        'F1 Macro': f1_macro,
        'F1 Micro': f1_micro,
        'F1 Weighted': f1_weight,
        'Balanced Accuracy': bal_acc,
        'Precision Macro': prec_macro,
        'Recall Macro': rec_macro,

        # ----- CE + proporções por classe -----
        'Cross-Entropy C0': ce_per_class[0],
        'Proporção C0':     prop_per_class[0],
        'Cross-Entropy C1': ce_per_class[1],
        'Proporção C1':     prop_per_class[1],
        'Cross-Entropy C2': ce_per_class[2],
        'Proporção C2':     prop_per_class[2],

        # ----- acurácia por classe -----
        'Accuracy_0': Acc_class[0],
        'Accuracy_1': Acc_class[1],
        'Accuracy_2': Acc_class[2],

        # ----- TP/FP/TN/FN por classe -----
        'TP_0': TP[0], 'FP_0': FP[0], 'TN_0': TN[0], 'FN_0': FN[0],
        'TP_1': TP[1], 'FP_1': FP[1], 'TN_1': TN[1], 'FN_1': FN[1],
        'TP_2': TP[2], 'FP_2': FP[2], 'TN_2': TN[2], 'FN_2': FN[2],

        # ----- TP/FP/TN/FN (%) por classe -----
        'TP_0(%)': TP_pct[0], 'FP_0(%)': FP_pct[0], 'TN_0(%)': TN_pct[0], 'FN_0(%)': FN_pct[0],
        'TP_1(%)': TP_pct[1], 'FP_1(%)': FP_pct[1], 'TN_1(%)': TN_pct[1], 'FN_1(%)': FN_pct[1],
        'TP_2(%)': TP_pct[2], 'FP_2(%)': FP_pct[2], 'TN_2(%)': TN_pct[2], 'FN_2(%)': FN_pct[2],
    }

# =========================
# Coleta de métricas por fold do CV interno (8 folds em df_tr)
# =========================
inner_rows = []
kf_inner = StratifiedKFold(n_splits=N_INNER_FOLDS, shuffle=True, random_state=SEED)
X80 = df_tr[FEATURES]; y80 = df_tr['y']

for fold, (tr_idx, te_idx) in enumerate(kf_inner.split(X80, y80), 1):
    Xtr_raw = X80.iloc[tr_idx].copy()
    Xte_raw = X80.iloc[te_idx].copy()
    y_tr, y_te = y80.iloc[tr_idx], y80.iloc[te_idx]

    X_tr_enc, X_te_enc = _fit_transform_fold(
        Xtr_raw,
        Xte_raw,
        VARIABLE_DISCRETIZE if 'VARIABLE_DISCRETIZE' in globals() else []
    )

    model = RandomForestClassifier(**rf_params)
    model.fit(X_tr_enc, y_tr)

    proba_te = model.predict_proba(X_te_enc)   # shape (n_amostras, 3)
    pred_te  = np.argmax(proba_te, axis=1)     # rótulo com maior probabilidade

    inner_rows.append(
        _compute_metrics_row('inner_cv', fold, y_te, pred_te, proba_mat=proba_te)
    )

pd.DataFrame(inner_rows).to_csv(CSV_OUT / 'cv_811_inner_folds.csv', index=False)

# =========================
# Avaliação em Val (fold 9) e Test (fold 10) com melhor rf_params
# =========================
def _fit_on_train_transform_three(Xtr_raw, Xva_raw, Xts_raw):
    # mapeia + encoder fitado no treino
    Xtr_m = _apply_fixed_mapping_referral(Xtr_raw)
    Xva_m = _apply_fixed_mapping_referral(Xva_raw)
    Xts_m = _apply_fixed_mapping_referral(Xts_raw)
    cat_cols, num_cols = _split_cat_num_columns(
        Xtr_m,
        VARIABLE_DISCRETIZE if 'VARIABLE_DISCRETIZE' in globals() else []
    )
    if len(cat_cols) > 0:
        enc = OrdinalEncoder(handle_unknown='use_encoded_value', unknown_value=-1, dtype=np.int64)
        Xtr_cat = enc.fit_transform(Xtr_m[cat_cols])
        Xva_cat = enc.transform(Xva_m[cat_cols])
        Xts_cat = enc.transform(Xts_m[cat_cols])

        def _rebuild(Xm, Xcat):
            return pd.concat(
                [
                    Xm[num_cols].reset_index(drop=True),
                    pd.DataFrame(Xcat, columns=cat_cols, index=Xm.index).reset_index(drop=True)
                ],
                axis=1
            )

        Xtr_enc = _rebuild(Xtr_m, Xtr_cat)
        Xva_enc = _rebuild(Xva_m, Xva_cat)
        Xts_enc = _rebuild(Xts_m, Xts_cat)
    else:
        Xtr_enc, Xva_enc, Xts_enc = Xtr_m.copy(), Xva_m.copy(), Xts_m.copy()
    return Xtr_enc, Xva_enc, Xts_enc

# --- prepara blocos (8 folds treino, 1 val, 1 teste) ---
Xtr_raw = df_tr[FEATURES].copy()
Xva_raw = df_va[FEATURES].copy()
Xts_raw = df_ts[FEATURES].copy()
y_tr = df_tr['y'].copy()
y_va = df_va['y'].copy()
y_ts = df_ts['y'].copy()

# encode com fit no treino (8 folds)
Xtr_enc, Xva_enc, Xts_enc = _fit_on_train_transform_three(Xtr_raw, Xva_raw, Xts_raw)

# --- modelo treinado só nos 8 folds (para avaliar fold 9) ---
best_model_val = RandomForestClassifier(**rf_params)
best_model_val.fit(Xtr_enc, y_tr)

# ---- VAL (fold 9) -> etapa = 'validacao', fold = 9
proba_va = best_model_val.predict_proba(Xva_enc)
pred_va  = np.argmax(proba_va, axis=1)

val_row = _compute_metrics_row(
    'validacao',
    9,
    y_va,
    pred_va,
    proba_mat=proba_va
)
pd.DataFrame([val_row]).to_csv(CSV_OUT / 'cv_811_validation.csv', index=False)

# --- agora junta folds 1–9 (treino + val) e treina de novo para teste final ---
train_1a9_idx = np.concatenate(fold_indices[0:9])
df_train_1a9 = df.iloc[train_1a9_idx].reset_index(drop=True)

Xtrain_final_raw = df_train_1a9[FEATURES].copy()
ytrain_final     = df_train_1a9['y'].copy()
Xtest_final_raw  = df_ts[FEATURES].copy()
ytest_final      = df_ts['y'].copy()

# encoder novamente com fit em 1–9 (garante uso de toda amostra de treino)
Xtr_fin_m = _apply_fixed_mapping_referral(Xtrain_final_raw)
Xts_fin_m = _apply_fixed_mapping_referral(Xtest_final_raw)
cat_cols_fin, num_cols_fin = _split_cat_num_columns(
    Xtr_fin_m,
    VARIABLE_DISCRETIZE if 'VARIABLE_DISCRETIZE' in globals() else []
)
if len(cat_cols_fin) > 0:
    enc_fin = OrdinalEncoder(handle_unknown='use_encoded_value', unknown_value=-1, dtype=np.int64)
    Xtr_fin_cat = enc_fin.fit_transform(Xtr_fin_m[cat_cols_fin])
    Xts_fin_cat = enc_fin.transform(Xts_fin_m[cat_cols_fin])

    def _rebuild_fin(Xm, Xcat):
        return pd.concat(
            [
                Xm[num_cols_fin].reset_index(drop=True),
                pd.DataFrame(Xcat, columns=cat_cols_fin, index=Xm.index).reset_index(drop=True)
            ],
            axis=1
        )

    Xtrain_final_enc = _rebuild_fin(Xtr_fin_m, Xtr_fin_cat)
    Xtest_final_enc  = _rebuild_fin(Xts_fin_m, Xts_fin_cat)
else:
    Xtrain_final_enc, Xtest_final_enc = Xtr_fin_m.copy(), Xts_fin_m.copy()

best_model_final = RandomForestClassifier(**rf_params)
best_model_final.fit(Xtrain_final_enc, ytrain_final)

# ---- TEST (fold 10) -> etapa = 'teste', fold = 10
proba_ts = best_model_final.predict_proba(Xtest_final_enc)
pred_ts  = np.argmax(proba_ts, axis=1)

test_row = _compute_metrics_row(
    'teste',
    10,
    ytest_final,
    pred_ts,
    proba_mat=proba_ts
)
pd.DataFrame([test_row]).to_csv(CSV_OUT / 'cv_811_teste.csv', index=False)

# =========================
# cv_811_results_long (8 folds + val + teste)
# =========================
results_long = inner_rows + [val_row, test_row]
pd.DataFrame(results_long).to_csv(CSV_OUT / 'cv_811_results_long.csv', index=False)

print("✅ Arquivos salvos em:", CSV_OUT.resolve())
print(" - cv_811_inner_folds.csv  (folds 1–8, etapa inner_cv)")
print(" - cv_811_validation.csv   (fold 9, etapa validacao)")
print(" - cv_811_teste.csv        (fold 10, etapa teste)")
print(" - cv_811_results_long.csv (todos juntos)")


[I 2026-01-26 04:41:56,685] A new study created in memory with name: rf_tuning_nested_811


📐 Esquema nested 10 folds:
 - Folds 1–8  → tuning interno (Optuna, CV 8-fold)
 - Fold 9     → validação externa (fold=9)
 - Fold 10    → teste hold-out final (fold=10)
🚀 Optuna nested (8 folds internos + 1 val + 1 teste): 3 trials × 8 folds internos


Best trial: 0. Best value: 0.98349:  33%|███▎      | 1/3 [00:33<01:07, 33.53s/it]

[I 2026-01-26 04:42:30,223] Trial 0 finished with value: 0.9834898635909082 and parameters: {'n_estimators': 1000, 'max_depth': 13, 'min_samples_split': 14, 'min_samples_leaf': 4, 'bootstrap': True, 'class_weight': None, 'max_features_mode': 'fraction', 'max_features': 0.30000000000000004, 'max_samples': 0.9847923138822793}. Best is trial 0 with value: 0.9834898635909082.


Best trial: 0. Best value: 0.98349:  67%|██████▋   | 2/3 [01:20<00:41, 41.33s/it]

[I 2026-01-26 04:43:17,021] Trial 1 finished with value: 0.9696276166113804 and parameters: {'n_estimators': 1650, 'max_depth': 20, 'min_samples_split': 5, 'min_samples_leaf': 22, 'bootstrap': True, 'class_weight': None, 'max_features_mode': 'sqrt', 'max_samples': 0.5157145928433671}. Best is trial 0 with value: 0.9834898635909082.


Best trial: 0. Best value: 0.98349: 100%|██████████| 3/3 [02:20<00:00, 46.70s/it]


[I 2026-01-26 04:44:16,800] Trial 2 finished with value: 0.9791889639634372 and parameters: {'n_estimators': 1450, 'max_depth': 40, 'min_samples_split': 29, 'min_samples_leaf': 8, 'bootstrap': True, 'class_weight': 'balanced_subsample', 'max_features_mode': 'sqrt', 'max_samples': 0.9541329429833268}. Best is trial 0 with value: 0.9834898635909082.
✅ Melhor valor (média AUC CV-8-fold em folds 1–8): 0.9834898635909082
✅ Melhor conjunto de hiperparâmetros (raw):
{'n_estimators': 1000, 'max_depth': 13, 'min_samples_split': 14, 'min_samples_leaf': 4, 'bootstrap': True, 'class_weight': None, 'max_features_mode': 'fraction', 'max_features': 0.30000000000000004, 'max_samples': 0.9847923138822793}

📦 rf_params para usar no pipeline:
{'n_jobs': -1, 'random_state': 42, 'n_estimators': 1000, 'max_depth': 13, 'min_samples_split': 14, 'min_samples_leaf': 4, 'bootstrap': True, 'class_weight': None, 'max_features': 0.30000000000000004, 'max_samples': 0.9847923138822793}
✅ Arquivos salvos em: C:\5_xAI\

In [6]:
# ================== OPTUNA (TPE + PRUNER) — nested 8/1/1 com 10 folds ==================
import optuna
from optuna.samplers import TPESampler
from optuna.pruners import MedianPruner
from optuna.exceptions import TrialPruned

from pathlib import Path
import numpy as np
import pandas as pd

from sklearn.model_selection import StratifiedKFold
from sklearn.metrics import (
    roc_auc_score, balanced_accuracy_score,
    accuracy_score, f1_score, precision_score, recall_score, confusion_matrix
)
from sklearn.ensemble import RandomForestClassifier
from sklearn.preprocessing import OrdinalEncoder

# --------- Saídas ---------
if 'CSV_OUT' not in globals():
    CSV_OUT = Path('./cv_out')
CSV_OUT.mkdir(parents=True, exist_ok=True)

# --------- Helpers já existentes (fallbacks) ----------
if '_apply_fixed_mapping_referral' not in globals():
    def _apply_fixed_mapping_referral(df_in):
        if 'referral_source' in df_in.columns and 'VARIABLE_DISCRETIZE_NUMBER' in globals():
            mapped = df_in['referral_source'].map(VARIABLE_DISCRETIZE_NUMBER)
            df_in = df_in.copy()
            df_in['referral_source'] = mapped.fillna(-1).astype('int64')
        return df_in

if '_split_cat_num_columns' not in globals():
    def _split_cat_num_columns(X, variable_discretize):
        non_numeric = [c for c in X.columns if not pd.api.types.is_numeric_dtype(X[c])]
        extra = [c for c in variable_discretize if (c in X.columns and not pd.api.types.is_numeric_dtype(X[c]))] \
                if 'VARIABLE_DISCRETIZE' in globals() else []
        cat_cols = sorted(list(dict.fromkeys(non_numeric + extra)))
        num_cols = [c for c in X.columns if c not in cat_cols]
        return cat_cols, num_cols

if '_fit_transform_fold' not in globals():
    def _fit_transform_fold(Xtr, Xte, variable_discretize):
        Xtr_m = _apply_fixed_mapping_referral(Xtr)
        Xte_m = _apply_fixed_mapping_referral(Xte)
        cat_cols, num_cols = _split_cat_num_columns(
            Xtr_m,
            variable_discretize if 'VARIABLE_DISCRETIZE' in globals() else []
        )
        if len(cat_cols) > 0:
            enc = OrdinalEncoder(handle_unknown='use_encoded_value', unknown_value=-1, dtype=np.int64)
            Xtr_cat = enc.fit_transform(Xtr_m[cat_cols])
            Xte_cat = enc.transform(Xte_m[cat_cols])
            Xtr_enc = pd.concat(
                [
                    Xtr_m[num_cols].reset_index(drop=True),
                    pd.DataFrame(Xtr_cat, columns=cat_cols, index=Xtr_m.index).reset_index(drop=True)
                ],
                axis=1
            )
            Xte_enc = pd.concat(
                [
                    Xte_m[num_cols].reset_index(drop=True),
                    pd.DataFrame(Xte_cat, columns=cat_cols, index=Xte_m.index).reset_index(drop=True)
                ],
                axis=1
            )
        else:
            Xtr_enc, Xte_enc = Xtr_m.copy(), Xte_m.copy()

        # sanity
        if Xtr_enc.isna().any().any():
            raise ValueError(f"Há NaN em X_train codificado: {Xtr_enc.columns[Xtr_enc.isna().any()].tolist()}")
        if Xte_enc.isna().any().any():
            raise ValueError(f"Há NaN em X_test codificado: {Xte_enc.columns[Xte_enc.isna().any()].tolist()}")
        return Xtr_enc, Xte_enc

# --------- Configurações ----------
SEED          = 42
N_TRIALS      = 3
N_OUTER_FOLDS = 10
N_INNER_FOLDS = 8

# FEATURES (garantia)
if 'FEATURES' not in globals():
    exclude_lower = {c.lower() for c in EXCLUDE_COLUMNS} if 'EXCLUDE_COLUMNS' in globals() else set()
    ID_COLS = [c for c in df.columns if c.lower() in exclude_lower]
    ID_COLS = [c for c in ID_COLS if c != 'y']
    FEATURES = [c for c in df.columns if c not in ID_COLS + ['y']]

y = df['y']
classes_ordenadas = np.sort(pd.unique(y))
N_CLASSES = int(pd.Series(y).nunique())

# ===============================
# Construção dos 10 folds estratificados (8/1/1 fixo)
# ===============================
outer_kfold = StratifiedKFold(n_splits=N_OUTER_FOLDS, shuffle=True, random_state=SEED)

fold_indices = []
for _, test_idx in outer_kfold.split(df[FEATURES], y):
    fold_indices.append(test_idx)

if len(fold_indices) != 10:
    raise RuntimeError(f"Esperados 10 folds, obtido {len(fold_indices)}")

train_cv_idx = np.concatenate(fold_indices[0:8])  # folds 1..8
val_ext_idx  = fold_indices[8]                    # fold 9
test_idx     = fold_indices[9]                    # fold 10

df_tr = df.iloc[train_cv_idx].reset_index(drop=True)
df_va = df.iloc[val_ext_idx].reset_index(drop=True)
df_ts = df.iloc[test_idx].reset_index(drop=True)

print(f"📐 Esquema nested 10 folds:")
print(f" - Folds 1–8  → tuning interno (Optuna, CV {N_INNER_FOLDS}-fold)")
print(f" - Fold 9     → validação externa (fold=9)")
print(f" - Fold 10    → teste hold-out final (fold=10)")
print(f" - Classes: {classes_ordenadas} | n_classes={N_CLASSES}")

# =========================
# Objetivo (CV interno em df_tr = 8 folds)
# =========================
def _suggest_params(trial: optuna.Trial):
    params = dict(
        n_estimators      = trial.suggest_int('n_estimators', 400, 2000, step=50),
        max_depth         = trial.suggest_categorical('max_depth', [None] + list(range(4, 41))),
        min_samples_split = trial.suggest_int('min_samples_split', 2, 30),
        min_samples_leaf  = trial.suggest_int('min_samples_leaf', 1, 30),
        bootstrap         = trial.suggest_categorical('bootstrap', [True, False]),
        class_weight      = trial.suggest_categorical('class_weight', [None, 'balanced', 'balanced_subsample']),
        n_jobs            = -1,
        random_state      = SEED
    )
    mf_mode = trial.suggest_categorical('max_features_mode', ['sqrt', 'log2', 'fraction'])
    if mf_mode == 'fraction':
        params['max_features'] = trial.suggest_float('max_features', 0.2, 0.9, step=0.05)
    else:
        params['max_features'] = mf_mode
    if params['bootstrap']:
        params['max_samples'] = trial.suggest_float('max_samples', 0.5, 1.0)
    return params

def _make_objective(features):
    kf = StratifiedKFold(n_splits=N_INNER_FOLDS, shuffle=True, random_state=SEED)
    X80 = df_tr[features]; y80 = df_tr['y']

    def objective(trial: optuna.Trial) -> float:
        params = _suggest_params(trial)
        fold_scores = []

        for fold, (tr_idx, te_idx) in enumerate(kf.split(X80, y80), 1):
            Xtr_raw = X80.iloc[tr_idx].copy()
            Xte_raw = X80.iloc[te_idx].copy()
            y_tr, y_te = y80.iloc[tr_idx], y80.iloc[te_idx]

            X_tr_enc, X_te_enc = _fit_transform_fold(
                Xtr_raw, Xte_raw,
                VARIABLE_DISCRETIZE if 'VARIABLE_DISCRETIZE' in globals() else []
            )

            model = RandomForestClassifier(**params)
            model.fit(X_tr_enc, y_tr)

            proba = model.predict_proba(X_te_enc)
            pred  = model.classes_[np.argmax(proba, axis=1)]  # <<< correto (sem bug)

            # score = AUC binário OU AUC OVR macro (3 classes)
            try:
                if pd.Series(y).nunique() == 2:
                    # classe positiva = 1 se existir, senão usa a segunda coluna
                    pos_idx = list(model.classes_).index(1) if 1 in model.classes_ else 1
                    score = roc_auc_score(y_te, proba[:, pos_idx])
                else:
                    # AUC multiclasse OVR macro (RECOMENDADO)
                    score = roc_auc_score(y_te, proba, multi_class='ovr', average='macro')
            except Exception:
                score = balanced_accuracy_score(y_te, pred)

            fold_scores.append(float(score))
            trial.report(float(score), step=fold)
            if trial.should_prune():
                raise TrialPruned()

        return float(np.mean(fold_scores))

    return objective

# =========================
# Execução do estudo (tuning nos folds 1–8)
# =========================
study = optuna.create_study(
    direction='maximize',
    sampler=TPESampler(seed=SEED),
    pruner=MedianPruner(n_startup_trials=20, n_warmup_steps=3),
    study_name='rf_tuning_nested_811'
)
print(f"🚀 Optuna nested (8 folds internos + 1 val + 1 teste): {N_TRIALS} trials × {N_INNER_FOLDS} folds internos")
study.optimize(_make_objective(FEATURES), n_trials=N_TRIALS, show_progress_bar=True)

print("✅ Melhor valor (média CV interna nos folds 1–8):", study.best_value)
print("✅ Melhor best_params (raw):", study.best_params)

# --------- Normalização das chaves auxiliares -> rf_params final ----------
_best = study.best_params.copy()
mf_mode = _best.pop('max_features_mode', None)
if mf_mode != 'fraction':
    _best['max_features'] = mf_mode
rf_params = dict(n_jobs=-1, random_state=SEED, **_best)
print("\n📦 rf_params para usar no pipeline:")
print(rf_params)

# =========================
# Helpers AUC/CE alinhados com model.classes_
# =========================
def _auc_ovr_macro(y_true, proba_mat, classes_order):
    """
    AUC multiclasse OVR macro (RECOMENDADO).
    - y_true: vetor com labels reais (ex: 0,1,2)
    - proba_mat: probas na ordem de classes_order (model.classes_)
    """
    y_true = np.asarray(y_true)
    proba_mat = np.asarray(proba_mat)
    try:
        if len(classes_order) == 2:
            pos_idx = list(classes_order).index(1) if 1 in classes_order else 1
            return float(roc_auc_score(y_true, proba_mat[:, pos_idx]))
        return float(roc_auc_score(y_true, proba_mat, multi_class='ovr', average='macro'))
    except Exception:
        return np.nan

def _cross_entropy_per_class_aligned(y_true, proba_mat, classes_order, eps=1e-15):
    """
    CE-Ck = média -log(p_k) nas amostras da classe k,
    alinhando p_k pela coluna correta via classes_order.
    """
    y_true = np.asarray(y_true)
    proba_mat = np.asarray(proba_mat)
    ce = {0: np.nan, 1: np.nan, 2: np.nan}

    for k in [0, 1, 2]:
        if k not in classes_order:
            continue
        col = list(classes_order).index(k)
        mask = (y_true == k)
        if mask.any():
            p = np.clip(proba_mat[mask, col], eps, 1.0 - eps)
            ce[k] = float(-np.mean(np.log(p)))
    return ce

# =========================
# Helper: monta linha de métricas (inclui AUC OVR macro)
# =========================
def _compute_metrics_row(etapa, fold, y_true, y_pred, proba_mat, classes_order):
    """
    etapa: 'inner_cv', 'validacao', 'teste'
    fold : 1..8 (inner), 9 (val), 10 (teste)
    y_true, y_pred: rótulos reais e previstos (labels originais)
    proba_mat: matriz (n_amostras, n_classes) na ordem classes_order (model.classes_)
    """
    y_true = np.asarray(y_true)
    y_pred = np.asarray(y_pred)
    proba_mat = np.asarray(proba_mat)
    total = len(y_true)

    labels = np.array(classes_ordenadas)  # usa as classes do dataset

    # ---------- CE por classe (0,1,2) alinhado ----------
    ce = _cross_entropy_per_class_aligned(y_true, proba_mat, classes_order)

    # ---------- Matriz de confusão (labels corretos) ----------
    cm = confusion_matrix(y_true, y_pred, labels=labels)

    # ---------- TP% FP% TN% FN% (macro OVR) ----------
    tps, fps, tns, fns = [], [], [], []
    for i, _ in enumerate(labels):
        tp = cm[i, i]
        fp = cm[:, i].sum() - tp
        fn = cm[i, :].sum() - tp
        tn = cm.sum() - (tp + fp + fn)
        tps.append(tp / total if total else np.nan)
        fps.append(fp / total if total else np.nan)
        tns.append(tn / total if total else np.nan)
        fns.append(fn / total if total else np.nan)

    # ---------- Métricas globais ----------
    acc_global = float(accuracy_score(y_true, y_pred))
    bal_acc    = float(balanced_accuracy_score(y_true, y_pred))
    f1_macro   = float(f1_score(y_true, y_pred, average='macro', zero_division=0))
    f1_micro   = float(f1_score(y_true, y_pred, average='micro', zero_division=0))
    f1_weight  = float(f1_score(y_true, y_pred, average='weighted', zero_division=0))
    prec_macro = float(precision_score(y_true, y_pred, average='macro', zero_division=0))
    rec_macro  = float(recall_score(y_true, y_pred, average='macro', zero_division=0))

    # ---------- AUC (binário) ou AUC OVR macro (multiclasse) ----------
    auc_val = _auc_ovr_macro(y_true, proba_mat, classes_order)

    return {
        'etapa': etapa,
        'fold': fold,

        # ----- globais -----
        'Acurácia': acc_global,
        'Balanced Accuracy': bal_acc,
        'AUC_OVR_MACRO': auc_val,  # <<< AQUI está o AUC que você quer

        'F1 Macro': f1_macro,
        'F1 Micro': f1_micro,
        'F1 Weighted': f1_weight,
        'Precision Macro': prec_macro,
        'Recall Macro': rec_macro,

        # ----- CE por classe (0/1/2) -----
        'Cross-Entropy C0': ce[0],
        'Cross-Entropy C1': ce[1],
        'Cross-Entropy C2': ce[2],

        # ----- taxas macro -----
        'TP%': float(np.nanmean(tps)),
        'FP%': float(np.nanmean(fps)),
        'TN%': float(np.nanmean(tns)),
        'FN%': float(np.nanmean(fns)),
    }

# =========================
# Coleta de métricas por fold do CV interno (8 folds em df_tr)
# =========================
inner_rows = []
kf_inner = StratifiedKFold(n_splits=N_INNER_FOLDS, shuffle=True, random_state=SEED)
X80 = df_tr[FEATURES]; y80 = df_tr['y']

for fold, (tr_idx, te_idx) in enumerate(kf_inner.split(X80, y80), 1):
    Xtr_raw = X80.iloc[tr_idx].copy()
    Xte_raw = X80.iloc[te_idx].copy()
    y_tr, y_te = y80.iloc[tr_idx], y80.iloc[te_idx]

    X_tr_enc, X_te_enc = _fit_transform_fold(
        Xtr_raw, Xte_raw,
        VARIABLE_DISCRETIZE if 'VARIABLE_DISCRETIZE' in globals() else []
    )

    model = RandomForestClassifier(**rf_params)
    model.fit(X_tr_enc, y_tr)

    proba_te = model.predict_proba(X_te_enc)
    pred_te  = model.classes_[np.argmax(proba_te, axis=1)]  # <<< correto

    inner_rows.append(
        _compute_metrics_row('inner_cv', fold, y_te, pred_te, proba_te, classes_order=model.classes_)
    )

pd.DataFrame(inner_rows).to_csv(CSV_OUT / 'cv_811_inner_folds.csv', index=False)

# =========================
# Avaliação em Val (fold 9) e Test (fold 10)
# =========================
def _fit_on_train_transform_three(Xtr_raw, Xva_raw, Xts_raw):
    Xtr_m = _apply_fixed_mapping_referral(Xtr_raw)
    Xva_m = _apply_fixed_mapping_referral(Xva_raw)
    Xts_m = _apply_fixed_mapping_referral(Xts_raw)
    cat_cols, num_cols = _split_cat_num_columns(
        Xtr_m,
        VARIABLE_DISCRETIZE if 'VARIABLE_DISCRETIZE' in globals() else []
    )
    if len(cat_cols) > 0:
        enc = OrdinalEncoder(handle_unknown='use_encoded_value', unknown_value=-1, dtype=np.int64)
        Xtr_cat = enc.fit_transform(Xtr_m[cat_cols])
        Xva_cat = enc.transform(Xva_m[cat_cols])
        Xts_cat = enc.transform(Xts_m[cat_cols])

        def _rebuild(Xm, Xcat):
            return pd.concat(
                [
                    Xm[num_cols].reset_index(drop=True),
                    pd.DataFrame(Xcat, columns=cat_cols, index=Xm.index).reset_index(drop=True)
                ],
                axis=1
            )

        Xtr_enc = _rebuild(Xtr_m, Xtr_cat)
        Xva_enc = _rebuild(Xva_m, Xva_cat)
        Xts_enc = _rebuild(Xts_m, Xts_cat)
    else:
        Xtr_enc, Xva_enc, Xts_enc = Xtr_m.copy(), Xva_m.copy(), Xts_m.copy()
    return Xtr_enc, Xva_enc, Xts_enc

# --- prepara blocos (8 folds treino, 1 val, 1 teste) ---
Xtr_raw = df_tr[FEATURES].copy()
Xva_raw = df_va[FEATURES].copy()
Xts_raw = df_ts[FEATURES].copy()
y_tr = df_tr['y'].copy()
y_va = df_va['y'].copy()
y_ts = df_ts['y'].copy()

Xtr_enc, Xva_enc, Xts_enc = _fit_on_train_transform_three(Xtr_raw, Xva_raw, Xts_raw)

# --- VAL (fold 9) ---
best_model_val = RandomForestClassifier(**rf_params)
best_model_val.fit(Xtr_enc, y_tr)

proba_va = best_model_val.predict_proba(Xva_enc)
pred_va  = best_model_val.classes_[np.argmax(proba_va, axis=1)]

val_row = _compute_metrics_row(
    'validacao', 9, y_va, pred_va, proba_va, classes_order=best_model_val.classes_
)
pd.DataFrame([val_row]).to_csv(CSV_OUT / 'cv_811_validation.csv', index=False)

# --- TEST (fold 10): treina em 1–9 e testa fold 10 ---
train_1a9_idx = np.concatenate(fold_indices[0:9])
df_train_1a9 = df.iloc[train_1a9_idx].reset_index(drop=True)

Xtrain_final_raw = df_train_1a9[FEATURES].copy()
ytrain_final     = df_train_1a9['y'].copy()
Xtest_final_raw  = df_ts[FEATURES].copy()
ytest_final      = df_ts['y'].copy()

Xtr_fin, Xts_fin = _fit_transform_fold(
    Xtrain_final_raw, Xtest_final_raw,
    VARIABLE_DISCRETIZE if 'VARIABLE_DISCRETIZE' in globals() else []
)

best_model_final = RandomForestClassifier(**rf_params)
best_model_final.fit(Xtr_fin, ytrain_final)

proba_ts = best_model_final.predict_proba(Xts_fin)
pred_ts  = best_model_final.classes_[np.argmax(proba_ts, axis=1)]

test_row = _compute_metrics_row(
    'teste', 10, ytest_final, pred_ts, proba_ts, classes_order=best_model_final.classes_
)
pd.DataFrame([test_row]).to_csv(CSV_OUT / 'cv_811_teste.csv', index=False)

# =========================
# cv_811_results_long (8 folds + val + teste)
# =========================
results_long = inner_rows + [val_row, test_row]
pd.DataFrame(results_long).to_csv(CSV_OUT / 'cv_811_results_long.csv', index=False)

print("✅ Arquivos salvos em:", CSV_OUT.resolve())
print(" - cv_811_inner_folds.csv  (folds 1–8, etapa inner_cv)")
print(" - cv_811_validation.csv   (fold 9, etapa validacao)")
print(" - cv_811_teste.csv        (fold 10, etapa teste)")
print(" - cv_811_results_long.csv (todos juntos)")

print("\n===== VALIDAÇÃO (AUC_OVR_MACRO) =====")
print("AUC_OVR_MACRO =", val_row["AUC_OVR_MACRO"])

print("\n===== TESTE (AUC_OVR_MACRO) =====")
print("AUC_OVR_MACRO =", test_row["AUC_OVR_MACRO"])


[I 2026-01-26 05:03:45,040] A new study created in memory with name: rf_tuning_nested_811


📐 Esquema nested 10 folds:
 - Folds 1–8  → tuning interno (Optuna, CV 8-fold)
 - Fold 9     → validação externa (fold=9)
 - Fold 10    → teste hold-out final (fold=10)
 - Classes: [0 1 2] | n_classes=3
🚀 Optuna nested (8 folds internos + 1 val + 1 teste): 3 trials × 8 folds internos


Best trial: 0. Best value: 0.98349:  33%|███▎      | 1/3 [00:34<01:08, 34.00s/it]

[I 2026-01-26 05:04:19,044] Trial 0 finished with value: 0.9834898635909082 and parameters: {'n_estimators': 1000, 'max_depth': 13, 'min_samples_split': 14, 'min_samples_leaf': 4, 'bootstrap': True, 'class_weight': None, 'max_features_mode': 'fraction', 'max_features': 0.30000000000000004, 'max_samples': 0.9847923138822793}. Best is trial 0 with value: 0.9834898635909082.


Best trial: 0. Best value: 0.98349:  67%|██████▋   | 2/3 [01:29<00:46, 46.63s/it]

[I 2026-01-26 05:05:14,508] Trial 1 finished with value: 0.9696276166113804 and parameters: {'n_estimators': 1650, 'max_depth': 20, 'min_samples_split': 5, 'min_samples_leaf': 22, 'bootstrap': True, 'class_weight': None, 'max_features_mode': 'sqrt', 'max_samples': 0.5157145928433671}. Best is trial 0 with value: 0.9834898635909082.


Best trial: 0. Best value: 0.98349: 100%|██████████| 3/3 [03:31<00:00, 70.65s/it]


[I 2026-01-26 05:07:16,970] Trial 2 finished with value: 0.9791889639634372 and parameters: {'n_estimators': 1450, 'max_depth': 40, 'min_samples_split': 29, 'min_samples_leaf': 8, 'bootstrap': True, 'class_weight': 'balanced_subsample', 'max_features_mode': 'sqrt', 'max_samples': 0.9541329429833268}. Best is trial 0 with value: 0.9834898635909082.
✅ Melhor valor (média CV interna nos folds 1–8): 0.9834898635909082
✅ Melhor best_params (raw): {'n_estimators': 1000, 'max_depth': 13, 'min_samples_split': 14, 'min_samples_leaf': 4, 'bootstrap': True, 'class_weight': None, 'max_features_mode': 'fraction', 'max_features': 0.30000000000000004, 'max_samples': 0.9847923138822793}

📦 rf_params para usar no pipeline:
{'n_jobs': -1, 'random_state': 42, 'n_estimators': 1000, 'max_depth': 13, 'min_samples_split': 14, 'min_samples_leaf': 4, 'bootstrap': True, 'class_weight': None, 'max_features': 0.30000000000000004, 'max_samples': 0.9847923138822793}


PermissionError: [Errno 13] Permission denied: '..\\model_reports\\random_forest\\cv\\csv\\cv_811_results_long.csv'

In [7]:
import datetime

# Gera um sufixo com hora para não dar erro de permissão
timestamp = datetime.datetime.now().strftime("%H%M%S")
novo_nome = f'cv_811_results_long_{timestamp}.csv'

try:
    # Tenta salvar no caminho original
    results_df = pd.DataFrame(results_long)
    results_df.to_csv(CSV_OUT / novo_nome, index=False)
    print(f"✅ Sucesso! Arquivo salvo como: {novo_nome}")
    print(f"Local: {CSV_OUT.resolve()}")
except Exception as e:
    print(f"❌ Erro ao salvar: {e}")


✅ Sucesso! Arquivo salvo como: cv_811_results_long_051202.csv
Local: C:\5_xAI\databases\UCI-THYROID-DXBIN\model_reports\random_forest\cv\csv


In [5]:
val_row

{'etapa': 'validacao',
 'fold': 9,
 'Acurácia': 0.9438902743142145,
 'F1 Macro': 0.8589814091298215,
 'F1 Micro': 0.9438902743142145,
 'F1 Weighted': 0.9405540969441669,
 'Balanced Accuracy': 0.8223094781219983,
 'Precision Macro': 0.9150263871059763,
 'Recall Macro': 0.8223094781219983,
 'Cross-Entropy C0': 0.06705878763511051,
 'Proporção C0': 0.8391521197007481,
 'Cross-Entropy C1': 0.3419961577863748,
 'Proporção C1': 0.07231920199501247,
 'Cross-Entropy C2': 0.9168388108359343,
 'Proporção C2': 0.0885286783042394,
 'Accuracy_0': 0.9451371571072319,
 'Accuracy_1': 0.9812967581047382,
 'Accuracy_2': 0.9613466334164589,
 'TP_0': 663,
 'FP_0': 34,
 'TN_0': 95,
 'FN_0': 10,
 'TP_1': 50,
 'FP_1': 7,
 'TN_1': 737,
 'FN_1': 8,
 'TP_2': 44,
 'FP_2': 4,
 'TN_2': 727,
 'FN_2': 27,
 'TP_0(%)': 0.8266832917705735,
 'FP_0(%)': 0.04239401496259352,
 'TN_0(%)': 0.11845386533665836,
 'FN_0(%)': 0.012468827930174564,
 'TP_1(%)': 0.06234413965087282,
 'FP_1(%)': 0.008728179551122194,
 'TN_1(%)': 0.9

In [ ]:
rf_params

In [ ]:
# ====================== RandomForest CV com OrdinalEncoder (binário e 3 classes) - 8/1/1 ======================
import os, time, numpy as np, pandas as pd
from pathlib import Path
from sklearn.model_selection import StratifiedKFold, StratifiedShuffleSplit
from sklearn.ensemble import RandomForestClassifier
from sklearn.metrics import (confusion_matrix, roc_curve, roc_auc_score, log_loss,
                             accuracy_score, f1_score, balanced_accuracy_score,
                             precision_score, recall_score)
from sklearn.preprocessing import OrdinalEncoder
from matplotlib.backends.backend_pdf import PdfPages
from tqdm import tqdm
import joblib

# ------------------------
# Preparação dos dados X, y
y = df['y']

# Detecta colunas identificadoras a partir de EXCLUDE_COLUMNS (case-insensitive)
exclude_lower = {c.lower() for c in EXCLUDE_COLUMNS}
ID_COLS = [c for c in df.columns if c.lower() in exclude_lower]
ID_COLS = [c for c in ID_COLS if c != 'y']  # garante que id columns não contenham a coluna alvo y

# FEATURES são todas as colunas exceto ids e y
FEATURES = [c for c in df.columns if c not in ID_COLS + ['y']]

# X_model será usado apenas para split (sem ids), sem normalizar e sem imputar
X_model_full = df[FEATURES].copy()
X_model_full.replace([np.inf, -np.inf], np.nan, inplace=True)

# --- Detectar número de classes e classes ordenadas ---
classes_ordenadas = np.sort(pd.unique(y))
n_classes = len(classes_ordenadas)

# ======================
# Split externo 8/1/1 (AGORA CONSISTENTE COM NESTED / HOLD-OUT)
# ======================
SEED = 42

if 'df_tr' in globals() and 'df_va' in globals() and 'df_ts' in globals():
    # ✅ Usa os mesmos splits definidos no Optuna nested:
    #    - df_tr : folds 1–8  (≈80%)
    #    - df_va : fold 9     (≈10%)
    #    - df_ts : fold 10    (≈10% hold-out)
    df_tr = df_tr.copy()
    df_va = df_va.copy()
    df_ts = df_ts.copy()

    if 'orig_index' not in df_tr.columns:
        df_tr = df_tr.reset_index(drop=False).rename(columns={'index': 'orig_index'})
    if 'orig_index' not in df_va.columns:
        df_va = df_va.reset_index(drop=False).rename(columns={'index': 'orig_index'})
    if 'orig_index' not in df_ts.columns:
        df_ts = df_ts.reset_index(drop=False).rename(columns={'index': 'orig_index'})

    print("📐 Usando df_tr/df_va/df_ts do Optuna nested (folds 1–8, 9, 10 — 8/1/1 com hold-out 10%).")

else:
    # 🔙 Fallback antigo: cria 8/1/1 via StratifiedShuffleSplit (caso nested não tenha sido rodado)
    sss_outer = StratifiedShuffleSplit(n_splits=1, test_size=0.2, random_state=SEED)
    tr_idx, tmp_idx = next(sss_outer.split(X_model_full, y))
    df_tr  = df.iloc[tr_idx].reset_index(drop=False).rename(columns={'index':'orig_index'})     # 80%
    df_tmp = df.iloc[tmp_idx].reset_index(drop=False).rename(columns={'index':'orig_index'})    # 20%

    sss_tmp = StratifiedShuffleSplit(n_splits=1, test_size=0.5, random_state=SEED)
    va_rel, ts_rel = next(sss_tmp.split(df_tmp[FEATURES], df_tmp['y']))
    df_va = df_tmp.iloc[va_rel].copy()   # 10%
    df_ts = df_tmp.iloc[ts_rel].copy()   # 10%

    print("📐 Split externo 8/1/1 criado via StratifiedShuffleSplit (fallback).")

# Dados para o CV interno (apenas 80% = df_tr)
X80 = df_tr[FEATURES].copy()
y80 = df_tr['y'].copy()

kf = StratifiedKFold(n_splits=N_SPLITS, shuffle=True, random_state=SEED)

# =========================
# (restante do código permanece IGUAL)
# =========================

# estruturas de resultado
resultados = []

# listas para métricas agregadas por fold (treino/teste)
aurocs_train = []
aurocs_test  = []
fprs_train = []
tprs_train = []
fprs_test  = []
tprs_test  = []

importancias_lista = []
X_train_parts = []
X_test_parts  = []
y_train_parts = []
y_test_parts  = []
y_train_pred_parts = []
y_test_pred_parts  = []

start_total = time.time()

# ------------------------
# Helpers

def _apply_fixed_mapping_referral(df_in):
    """Aplica mapeamento fixo para referral_source se existir na base."""
    if 'referral_source' in df_in.columns:
        mapped = df_in['referral_source'].map(VARIABLE_DISCRETIZE_NUMBER)
        df_in = df_in.copy()
        df_in['referral_source'] = mapped.fillna(-1).astype('int64')
    return df_in

def _split_cat_num_columns(X, variable_discretize):
    """Identifica colunas categóricas:
       - todas as não numéricas
       - + colunas em VARIABLE_DISCRETIZE que não sejam numericamente tipadas no DataFrame
    """
    non_numeric = [c for c in X.columns if not pd.api.types.is_numeric_dtype(X[c])]
    extra = [c for c in variable_discretize if (c in X.columns and not pd.api.types.is_numeric_dtype(X[c]))]
    cat_cols = sorted(list(dict.fromkeys(non_numeric + extra)))
    num_cols = [c for c in X.columns if c not in cat_cols]
    return cat_cols, num_cols

def _fit_transform_fold(Xtr, Xte, variable_discretize):
    """Prepara Xtr/Xte para o fold:
       1) mapeia referral_source com dicionário fixo
       2) ordinal-encode nas demais categóricas (fit no treino, transform no teste) com unknown=-1
    """
    Xtr_m = _apply_fixed_mapping_referral(Xtr)
    Xte_m = _apply_fixed_mapping_referral(Xte)

    cat_cols, num_cols = _split_cat_num_columns(Xtr_m, variable_discretize)

    if len(cat_cols) > 0:
        enc = OrdinalEncoder(handle_unknown='use_encoded_value', unknown_value=-1, dtype=np.int64)
        Xtr_cat = enc.fit_transform(Xtr_m[cat_cols])
        Xte_cat = enc.transform(Xte_m[cat_cols])

        Xtr_enc = pd.concat(
            [Xtr_m[num_cols].reset_index(drop=True),
             pd.DataFrame(Xtr_cat, columns=cat_cols, index=Xtr_m.index).reset_index(drop=True)],
            axis=1
        )
        Xte_enc = pd.concat(
            [Xte_m[num_cols].reset_index(drop=True),
             pd.DataFrame(Xte_cat, columns=cat_cols, index=Xte_m.index).reset_index(drop=True)],
            axis=1
        )
    else:
        Xtr_enc, Xte_enc = Xtr_m.copy(), Xte_m.copy()

    # sanity check: sem NaN
    if Xtr_enc.isna().any().any():
        cols = Xtr_enc.columns[Xtr_enc.isna().any()].tolist()
        raise ValueError(f"Há NaN em X_train codificado nas colunas: {cols}.")
    if Xte_enc.isna().any().any():
        cols = Xte_enc.columns[Xte_enc.isna().any()].tolist()
        raise ValueError(f"Há NaN em X_test codificado nas colunas: {cols}.")

    return Xtr_enc, Xte_enc

def _fit_on_train_transform_three(Xtr_raw, Xva_raw, Xts_raw):
    """Fit encoder no treino (80%) e transforma val/test (10%/10%) — sem normalizar/imputar."""
    Xtr_m = _apply_fixed_mapping_referral(Xtr_raw)
    Xva_m = _apply_fixed_mapping_referral(Xva_raw)
    Xts_m = _apply_fixed_mapping_referral(Xts_raw)
    cat_cols, num_cols = _split_cat_num_columns(Xtr_m, VARIABLE_DISCRETIZE)
    if len(cat_cols) > 0:
        enc = OrdinalEncoder(handle_unknown='use_encoded_value', unknown_value=-1, dtype=np.int64)
        Xtr_cat = enc.fit_transform(Xtr_m[cat_cols])
        Xva_cat = enc.transform(Xva_m[cat_cols])
        Xts_cat = enc.transform(Xts_m[cat_cols])

        def _rebuild(Xm, Xcat):
            return pd.concat(
                [Xm[num_cols].reset_index(drop=True),
                 pd.DataFrame(Xcat, columns=cat_cols, index=Xm.index).reset_index(drop=True)], axis=1
            )
        Xtr_enc = _rebuild(Xtr_m, Xtr_cat)
        Xva_enc = _rebuild(Xva_m, Xva_cat)
        Xts_enc = _rebuild(Xts_m, Xts_cat)
    else:
        Xtr_enc, Xva_enc, Xts_enc = Xtr_m.copy(), Xva_m.copy(), Xts_m.copy()
    return Xtr_enc, Xva_enc, Xts_enc

def _roc_ovr_curves(y_true, proba_matrix, classes):
    """Retorna dicionários fprs_por_classe, tprs_por_classe (apenas se houver 1+1 amostras por classe binária)."""
    fprs_cls, tprs_cls = {}, {}
    for i, cls in enumerate(classes):
        y_bin = (y_true == cls).astype(int)
        if y_bin.sum() > 0 and (len(y_bin) - y_bin.sum()) > 0:
            fpr, tpr, _ = roc_curve(y_bin, proba_matrix[:, i])
            fprs_cls[int(cls)] = fpr
            tprs_cls[int(cls)] = tpr
        else:
            fprs_cls[int(cls)] = None
            tprs_cls[int(cls)] = None
    return fprs_cls, tprs_cls

with PdfPages(PDF_OUT / f'RF_CV_{N_SPLITS}.pdf') as pdf:
    with tqdm(total=N_SPLITS, desc="🔄 Folds K-Fold (80%)") as pbar:
        for fold, (train_idx, test_idx) in enumerate(kf.split(X80, y80), 1):
            t0 = time.time()

            # Para manter ids nos arquivos de saída, indexamos sobre o DataFrame dos 80%
            df_tr_fold = df_tr.set_index('orig_index')  # garantir chave consistente

            X_train_full = df_tr.iloc[train_idx][ID_COLS + FEATURES] if len(ID_COLS)>0 else df_tr.iloc[train_idx][FEATURES]
            X_test_full  = df_tr.iloc[test_idx][ID_COLS + FEATURES]  if len(ID_COLS)>0 else df_tr.iloc[test_idx][FEATURES]

            # Usamos apenas as FEATURES (sem ids) para treinar o modelo
            X_train = X_train_full[FEATURES].copy()
            X_test  = X_test_full[FEATURES].copy()
            y_train, y_test = y80.iloc[train_idx], y80.iloc[test_idx]

            # === Codificação por fold (sem normalizar, sem imputar) ===
            X_train_enc, X_test_enc = _fit_transform_fold(X_train, X_test, VARIABLE_DISCRETIZE)

            # Treino
            model = RandomForestClassifier(**rf_params)
            model.fit(X_train_enc, y_train)

            # Previsões de probabilidade
            y_proba_test_all  = model.predict_proba(X_test_enc)
            y_proba_train_all = model.predict_proba(X_train_enc)

            # Colunas de probabilidade dinâmicas deste fold
            prob_cols = [f'prob_{int(c)}' for c in classes_ordenadas]

            if n_classes == 2:
                idx_c0 = np.where(classes_ordenadas == 0)[0][0]
                idx_c1 = np.where(classes_ordenadas == 1)[0][0]

                y_proba_test_C1  = y_proba_test_all[:, idx_c1]
                y_proba_train_C1 = y_proba_train_all[:, idx_c1]
                y_pred_test_bin  = (y_proba_test_C1  >= THRESHOLD).astype(int)
                y_pred_train_bin = (y_proba_train_C1 >= THRESHOLD).astype(int)

                # Guardar blocos/ids e probabilidades (prob_0/prob_1) — FOLDS DOS 80%
                X_test_parts.append(X_test_full.assign(fold=fold).reset_index())
                y_test_parts.append(pd.Series(y_test, name='y_test').to_frame().assign(fold=fold).reset_index())
                y_test_pred_parts.append(
                    pd.DataFrame(y_proba_test_all, columns=prob_cols, index=X_test_full.index)
                      .assign(fold=fold).reset_index()
                )

                X_train_parts.append(X_train_full.assign(fold=fold).reset_index())
                y_train_parts.append(pd.Series(y_train, name='y_train').to_frame().assign(fold=fold).reset_index())
                y_train_pred_parts.append(
                    pd.DataFrame(y_proba_train_all, columns=prob_cols, index=X_train_full.index)
                      .assign(fold=fold).reset_index()
                )

                # Métricas Treino (binárias)
                cm_train = confusion_matrix(y_train, y_pred_train_bin, labels=[0,1])
                tn_tr, fp_tr, fn_tr, tp_tr = cm_train.ravel()
                total_cm_treino = cm_train.sum()

                fpr_tr_dict, tpr_tr_dict = _roc_ovr_curves(y_train, y_proba_train_all, classes_ordenadas)
                fprs_train.append(fpr_tr_dict)
                tprs_train.append(tpr_tr_dict)

                auroc_tr = roc_auc_score(y_train, y_proba_train_C1)

                ce_train_0 = log_loss((y_train==0).astype(int), y_proba_train_all[:, idx_c0], labels=[0,1]) if len(y_train)>0 else np.nan
                ce_train_1 = log_loss((y_train==1).astype(int), y_proba_train_all[:, idx_c1], labels=[0,1]) if len(y_train)>0 else np.nan
                cross_prop_train = pd.Series(y_train).value_counts(normalize=True) * 100

                # Métricas Teste (binárias) — teste do FOLD (ainda dentro dos 80%)
                cm_test = confusion_matrix(y_test, y_pred_test_bin, labels=[0,1])
                tn_ts, fp_ts, fn_ts, tp_ts = cm_test.ravel()
                total_cm_teste = cm_test.sum()

                fpr_ts_dict, tpr_ts_dict = _roc_ovr_curves(y_test, y_proba_test_all, classes_ordenadas)
                fprs_test.append(fpr_ts_dict)
                tprs_test.append(tpr_ts_dict)

                auroc_ts  = roc_auc_score(y_test, y_proba_test_C1)
                ce_test_0 = log_loss((y_test==0).astype(int), y_proba_test_all[:, idx_c0], labels=[0,1]) if len(y_test)>0 else np.nan
                ce_test_1 = log_loss((y_test==1).astype(int), y_proba_test_all[:, idx_c1], labels=[0,1]) if len(y_test)>0 else np.nan
                cross_prop_test = pd.Series(y_test).value_counts(normalize=True) * 100

                aurocs_train.append(auroc_tr)
                aurocs_test.append(auroc_ts)

                resultados.append({
                    'Conjunto': 'Treino(80%)',
                    'Fold': fold,
                    'Acurácia': accuracy_score(y_train, y_pred_train_bin),
                    'Cross-Entropy C0': ce_train_0,
                    'Proporção C0': cross_prop_train.loc[0] if 0 in cross_prop_train.index else np.nan,
                    'Cross-Entropy C1': ce_train_1,
                    'Proporção C1': cross_prop_train.loc[1] if 1 in cross_prop_train.index else np.nan,
                    'Cross-Entropy C2': np.nan,
                    'Proporção C2': np.nan,
                    'F1 Score': f1_score(y_train, y_pred_train_bin),
                    'Balanced Accuracy': balanced_accuracy_score(y_train, y_pred_train_bin),
                    'Precision': precision_score(y_train, y_pred_train_bin),
                    'Recall': recall_score(y_train, y_pred_train_bin),
                    'AUROC': auroc_tr,
                    'TP': tp_tr, 'FP': fp_tr, 'TN': tn_tr, 'FN': fn_tr,
                    'TP(%)': round(tp_tr / total_cm_treino * 100, 2),
                    'FP(%)': round(fp_tr / total_cm_treino * 100, 2),
                    'TN(%)': round(tn_tr / total_cm_treino * 100, 2),
                    'FN(%)': round(fn_tr / total_cm_treino * 100, 2),
                    'Tempo (s)': round(time.time() - t0, 2)
                })

                resultados.append({
                    'Conjunto': 'TesteFold(80%)',
                    'Fold': fold,
                    'Acurácia': accuracy_score(y_test, y_pred_test_bin),
                    'Cross-Entropy C0': ce_test_0,
                    'Proporção C0': cross_prop_test.loc[0] if 0 in cross_prop_test.index else np.nan,
                    'Cross-Entropy C1': ce_test_1,
                    'Proporção C1': cross_prop_test.loc[1] if 1 in cross_prop_test.index else np.nan,
                    'Cross-Entropy C2': np.nan,
                    'Proporção C2': np.nan,
                    'F1 Score': f1_score(y_test, y_pred_test_bin),
                    'Balanced Accuracy': balanced_accuracy_score(y_test, y_pred_test_bin),
                    'Precision': precision_score(y_test, y_pred_test_bin),
                    'Recall': recall_score(y_test, y_pred_test_bin),
                    'AUROC': auroc_ts,
                    'TP': tp_ts, 'FP': fp_ts, 'TN': tn_ts, 'FN': fn_ts,
                    'TP(%)': round(tp_ts / total_cm_teste * 100, 2),
                    'FP(%)': round(fp_ts / total_cm_teste * 100, 2),
                    'TN(%)': round(tn_ts / total_cm_teste * 100, 2),
                    'FN(%)': round(fn_ts / total_cm_teste * 100, 2),
                    'Tempo (s)': round(time.time() - t0, 2)
                })

            else:
                # --- Multiclasse ---
                y_pred_test  = classes_ordenadas[np.argmax(y_proba_test_all, axis=1)]
                y_pred_train = classes_ordenadas[np.argmax(y_proba_train_all, axis=1)]

                X_test_parts.append(X_test_full.assign(fold=fold).reset_index())
                y_test_parts.append(pd.Series(y_test, name='y_test').to_frame().assign(fold=fold).reset_index())
                y_test_pred_parts.append(
                    pd.DataFrame(y_proba_test_all, columns=prob_cols, index=X_test_full.index)
                      .assign(fold=fold).reset_index()
                )

                X_train_parts.append(X_train_full.assign(fold=fold).reset_index())
                y_train_parts.append(pd.Series(y_train, name='y_train').to_frame().assign(fold=fold).reset_index())
                y_train_pred_parts.append(
                    pd.DataFrame(y_proba_train_all, columns=prob_cols, index=X_train_full.index)
                      .assign(fold=fold).reset_index()
                )

                ce_train = {}
                ce_test  = {}
                prop_train = pd.Series(y_train).value_counts(normalize=True) * 100
                prop_test  = pd.Series(y_test ).value_counts(normalize=True) * 100
                for i, cls in enumerate(classes_ordenadas):
                    ce_train[int(cls)] = log_loss((y_train==cls).astype(int), y_proba_train_all[:, i], labels=[0,1]) if len(y_train)>0 else np.nan
                    ce_test[int(cls)]  = log_loss((y_test==cls).astype(int),  y_proba_test_all[:,  i], labels=[0,1])  if len(y_test)>0  else np.nan

                try:
                    auroc_tr = roc_auc_score(y_train, y_proba_train_all, multi_class='ovr', average='macro')
                except Exception:
                    auroc_tr = np.nan
                try:
                    auroc_ts = roc_auc_score(y_test, y_proba_test_all, multi_class='ovr', average='macro')
                except Exception:
                    auroc_ts = np.nan

                aurocs_train.append(auroc_tr)
                aurocs_test.append(auroc_ts)

                fpr_tr_dict, tpr_tr_dict = _roc_ovr_curves(y_train, y_proba_train_all, classes_ordenadas)
                fpr_ts_dict, tpr_ts_dict = _roc_ovr_curves(y_test,  y_proba_test_all,  classes_ordenadas)
                fprs_train.append(fpr_tr_dict); tprs_train.append(tpr_tr_dict)
                fprs_test.append(fpr_ts_dict);  tprs_test.append(tpr_ts_dict)

                tp_tr = fp_tr = tn_tr = fn_tr = np.nan
                tp_ts = fp_ts = tn_ts = fn_ts = np.nan
                total_cm_treino = total_cm_teste = np.nan

                resultados.append({
                    'Conjunto': 'Treino(80%)',
                    'Fold': fold,
                    'Acurácia': accuracy_score(y_train, y_pred_train),
                    'Cross-Entropy C0': ce_train.get(0, np.nan),
                    'Proporção C0': prop_train.loc[0] if 0 in prop_train.index else np.nan,
                    'Cross-Entropy C1': ce_train.get(1, np.nan),
                    'Proporção C1': prop_train.loc[1] if 1 in prop_train.index else np.nan,
                    'Cross-Entropy C2': ce_train.get(2, np.nan),
                    'Proporção C2': prop_train.loc[2] if 2 in prop_train.index else np.nan,
                    'F1 Score': f1_score(y_train, y_pred_train, average='macro'),
                    'Balanced Accuracy': balanced_accuracy_score(y_train, y_pred_train),
                    'Precision': precision_score(y_train, y_pred_train, average='macro', zero_division=0),
                    'Recall': recall_score(y_train, y_pred_train, average='macro', zero_division=0),
                    'AUROC': auroc_tr,
                    'TP': tp_tr, 'FP': fp_tr, 'TN': tn_tr, 'FN': fn_tr,
                    'TP(%)': np.nan, 'FP(%)': np.nan, 'TN(%)': np.nan, 'FN(%)': np.nan,
                    'Tempo (s)': round(time.time() - t0, 2)
                })

                resultados.append({
                    'Conjunto': 'TesteFold(80%)',
                    'Fold': fold,
                    'Acurácia': accuracy_score(y_test, y_pred_test),
                    'Cross-Entropy C0': ce_test.get(0, np.nan),
                    'Proporção C0': prop_test.loc[0] if 0 in prop_test.index else np.nan,
                    'Cross-Entropy C1': ce_test.get(1, np.nan),
                    'Proporção C1': prop_test.loc[1] if 1 in prop_test.index else np.nan,
                    'Cross-Entropy C2': ce_test.get(2, np.nan),
                    'Proporção C2': prop_test.loc[2] if 2 in prop_test.index else np.nan,
                    'F1 Score': f1_score(y_test, y_pred_test, average='macro'),
                    'Balanced Accuracy': balanced_accuracy_score(y_test, y_pred_test),
                    'Precision': precision_score(y_test, y_pred_test, average='macro', zero_division=0),
                    'Recall': recall_score(y_test, y_pred_test, average='macro', zero_division=0),
                    'AUROC': auroc_ts,
                    'TP': tp_ts, 'FP': fp_ts, 'TN': tn_ts, 'FN': fn_ts,
                    'TP(%)': np.nan, 'FP(%)': np.nan, 'TN(%)': np.nan, 'FN(%)': np.nan,
                    'Tempo (s)': round(time.time() - t0, 2)
                })

            # Importância das features (percentual)
            importancias = model.feature_importances_
            nomes_colunas = FEATURES
            linha_importancia = {'Conjunto': 'TesteFold(80%)', 'Fold': fold}
            for nome, valor in zip(nomes_colunas, importancias):
                linha_importancia[nome] = f"{valor * 100:.2f}%"
            importancias_lista.append(linha_importancia)

            # salva modelo por fold (opcional)
            model_path = MODEL_DIR / f'rf_model_cf{N_SPLITS}.pkl'
            joblib.dump(model, model_path)

            pbar.update(1)

# ======================
# Pós-CV: salvar resultados e blocos (mantido)
# ======================
df_resultados = pd.DataFrame(resultados)

# ----- Avaliação adicional em Val(10%) e Test(10%) usando modelo treinado nos 80% -----
# Encoder fitado nos 80% e transform em Val/Test
Xtr_raw = df_tr[FEATURES].copy()
Xva_raw = df_va[FEATURES].copy()
Xts_raw = df_ts[FEATURES].copy()
y_tr = df_tr['y'].copy()
y_va = df_va['y'].copy()
y_ts = df_ts['y'].copy()

Xtr_enc, Xva_enc, Xts_enc = _fit_on_train_transform_three(Xtr_raw, Xva_raw, Xts_raw)

final80_model = RandomForestClassifier(**rf_params)
final80_model.fit(Xtr_enc, y_tr)

def _pack_bin_mc(y_true, proba_all, pred_all):
    if n_classes == 2:
        idx_c1 = np.where(classes_ordenadas == 1)[0][0]
        auc_ = roc_auc_score(y_true, proba_all[:, idx_c1])
        f1_  = f1_score(y_true, pred_all)
        pre_ = precision_score(y_true, pred_all)
        rec_ = recall_score(y_true, pred_all)
        bal_ = balanced_accuracy_score(y_true, pred_all)
        acc_ = accuracy_score(y_true, pred_all)
    else:
        auc_ = roc_auc_score(y_true, proba_all, multi_class='ovr', average='macro')
        f1_  = f1_score(y_true, pred_all, average='macro')
        pre_ = precision_score(y_true, pred_all, average='macro', zero_division=0)
        rec_ = recall_score(y_true, pred_all, average='macro', zero_division=0)
        bal_ = balanced_accuracy_score(y_true, pred_all)
        acc_ = accuracy_score(y_true, pred_all)
    return acc_, f1_, pre_, rec_, bal_, auc_

# Validação 10%
proba_va = final80_model.predict_proba(Xva_enc)
pred_va  = classes_ordenadas[np.argmax(proba_va, axis=1)]
acc_v, f1_v, pre_v, rec_v, bal_v, auc_v = _pack_bin_mc(y_va, proba_va, pred_va)
df_resultados = pd.concat([df_resultados, pd.DataFrame([{
    'Conjunto': 'Validação(10%)', 'Fold': 0,
    'Acurácia': acc_v, 'F1 Score': f1_v, 'Balanced Accuracy': bal_v,
    'Precision': pre_v, 'Recall': rec_v, 'AUROC': auc_v
}])], ignore_index=True)

# Teste 10%
proba_ts = final80_model.predict_proba(Xts_enc)
pred_ts  = classes_ordenadas[np.argmax(proba_ts, axis=1)]
acc_t, f1_t, pre_t, rec_t, bal_t, auc_t = _pack_bin_mc(y_ts, proba_ts, pred_ts)
df_resultados = pd.concat([df_resultados, pd.DataFrame([{
    'Conjunto': 'Teste(10%)', 'Fold': 0,
    'Acurácia': acc_t, 'F1 Score': f1_t, 'Balanced Accuracy': bal_t,
    'Precision': pre_t, 'Recall': rec_t, 'AUROC': auc_t
}])], ignore_index=True)

# adiciona média das métricas numéricas (mantido)
mean_row = df_resultados.select_dtypes(include=[np.number]).mean()
mean_row['Conjunto'] = 'Média'
mean_row['Fold'] = 'Média'
df_resultados = pd.concat([df_resultados, pd.DataFrame([mean_row])], ignore_index=True, sort=False)

# salva csvs principais (mantidos)
df_resultados.to_csv(CSV_OUT / f'rf_cv_results_{N_SPLITS}.csv', index=False)
pd.DataFrame(importancias_lista).to_csv(CSV_OUT / f'rf_feature_importances_{N_SPLITS}.csv', index=False)

# Concatena blocos para gerar dataframes completos de treino/teste (INTERNOS 80%) com probabilidades
if len(X_train_parts)>0:
    X_train_global = pd.concat(X_train_parts, ignore_index=True)
    y_train_global = pd.concat(y_train_parts, ignore_index=True)
    y_train_pred_global = pd.concat(y_train_pred_parts, ignore_index=True)
else:
    X_train_global = pd.DataFrame(); y_train_global = pd.DataFrame(); y_train_pred_global = pd.DataFrame()

if len(X_test_parts)>0:
    X_test_global = pd.concat(X_test_parts, ignore_index=True)
    y_test_global = pd.concat(y_test_parts, ignore_index=True)
    y_test_pred_global = pd.concat(y_test_pred_parts, ignore_index=True)
else:
    X_test_global = pd.DataFrame(); y_test_global = pd.DataFrame(); y_test_pred_global = pd.DataFrame()

# Ajusta 'index' -> 'orig_index' para merges
for df_block in [X_train_global, X_test_global, y_train_global, y_test_global, y_train_pred_global, y_test_pred_global]:
    if isinstance(df_block, pd.DataFrame) and ('index' in df_block.columns):
        df_block.rename(columns={'index': 'orig_index'}, inplace=True)

# Mescla para criar arquivos finais com probas e fold (INTERNOS 80%)
if not X_train_global.empty and not y_train_global.empty:
    train_df = X_train_global.merge(y_train_global, on=['orig_index', 'fold'], how='left')
    if not y_train_pred_global.empty:
        train_df = train_df.merge(y_train_pred_global, on=['orig_index', 'fold'], how='left')
else:
    train_df = pd.DataFrame()

if not X_test_global.empty and not y_test_global.empty:
    test_df = X_test_global.merge(y_test_global, on=['orig_index', 'fold'], how='left')
    if not y_test_pred_global.empty:
        test_df = test_df.merge(y_test_pred_global, on=['orig_index', 'fold'], how='left')
else:
    test_df = pd.DataFrame()

# Salva os datasets com probabilidades e y_pred (INTERNOS 80%)
if not test_df.empty:
    if n_classes == 2:
        test_df['y_proba'] = test_df['prob_1']
        test_df['y_pred']  = (test_df['y_proba'] >= THRESHOLD).astype(int)
    else:
        prob_cols_present = [c for c in test_df.columns if c.startswith('prob_')]
        test_df['y_pred']  = test_df[prob_cols_present].values.argmax(axis=1)
        test_df['y_proba'] = test_df[prob_cols_present].max(axis=1)
    test_df.to_csv(CSV_OUT / f'rf_test_with_proba_{N_SPLITS}.csv', index=False)

if not train_df.empty:
    if n_classes == 2:
        train_df['y_proba'] = train_df['prob_1']
        train_df['y_pred']  = (train_df['y_proba'] >= THRESHOLD).astype(int)
    else:
        prob_cols_present = [c for c in train_df.columns if c.startswith('prob_')]
        train_df['y_pred']  = train_df[prob_cols_present].values.argmax(axis=1)
        train_df['y_proba'] = train_df[prob_cols_present].max(axis=1)
    train_df.to_csv(CSV_OUT / f'rf_train_with_proba_{N_SPLITS}.csv', index=False)

# (mantido) Exporta um dataset "ajustado" (codificado) para reuso em outros momentos
try:
    df_m = _apply_fixed_mapping_referral(df)
    cat_cols_full, num_cols_full = _split_cat_num_columns(df_m[FEATURES], VARIABLE_DISCRETIZE)
    if len(cat_cols_full) > 0:
        enc_full = OrdinalEncoder(handle_unknown='use_encoded_value', unknown_value=-1, dtype=np.int64)
        X_full_cat = enc_full.fit_transform(df_m[FEATURES][cat_cols_full])
        X_full_enc = pd.concat(
            [df_m[FEATURES][num_cols_full].reset_index(drop=True),
             pd.DataFrame(X_full_cat, columns=cat_cols_full, index=df_m.index).reset_index(drop=True)],
            axis=1
        )
    else:
        X_full_enc = df_m[FEATURES].copy()

    df_encoded_full = pd.concat([df_m[ID_COLS].reset_index(drop=True),
                                 X_full_enc.reset_index(drop=True),
                                 y.reset_index(drop=True)], axis=1)

    if df_encoded_full.isna().any().any():
        cols = df_encoded_full.columns[df_encoded_full.isna().any()].tolist()
        raise ValueError(f"Há NaN no dataset codificado exportado nas colunas: {cols}.")

    out_proc = Path('../data/processed') / f'{DATASET_NAME}_encoded_ordinal.csv'
    df_encoded_full.to_csv(out_proc, index=False)
    print(f'Dataset ajustado (codificado) salvo em: {out_proc}')
except Exception as e_export:
    print('Falha ao exportar dataset ajustado (codificado):', e_export)

print('Resultados salvos em:', CSV_OUT)
# ===================================================================================


In [ ]:
# # Diagnóstico rápido para NaN / tipos em train_df e test_df
# for name, ycol in [('train_df', 'y_train'), ('test_df', 'y_test')]:
#     df_block = globals().get(name)
#     if df_block is None:
#         print(f"{name} não existe")
#         continue
#     if df_block.empty:
#         print(f"{name} existe mas está vazio")
#         continue
#     print(f"\n{name}: shape={df_block.shape}")
#     for col in [ycol, 'prob_1', 'y_proba']:
#         if col in df_block.columns:
#             print(f"  {col}: nulls={df_block[col].isnull().sum()}, dtype={df_block[col].dtype}, unique_count={df_block[col].nunique(dropna=True)}")
#     cols_to_show = [c for c in [ycol, 'prob_1', 'y_proba'] if c in df_block.columns]
#     display(df_block[cols_to_show].head(10))

In [ ]:
# =================== PDF RESUMO com suporte a 3 classes + ROC OvR e salvamento de imagens (compatível 8/1/1) ===================

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns
from matplotlib.backends.backend_pdf import PdfPages
from sklearn.metrics import roc_curve, auc, confusion_matrix

# Garante pasta de imagens
try:
    IMAGES_OUT
except NameError:
    IMAGES_OUT = PDF_OUT.parent / "images"
IMAGES_OUT.mkdir(parents=True, exist_ok=True)

# ---------- Helpers existentes ----------
def _get_classes_from_probs_columns(df_block):
    """Infere classes a partir das colunas prob_*. Retorna lista ordenada de ints e a lista de colunas prob_."""
    prob_cols = sorted([c for c in df_block.columns if c.startswith('prob_')], key=lambda s: int(s.split('_')[1]))
    classes = [int(c.split('_')[1]) for c in prob_cols]
    return classes, prob_cols

def _aggregate_mean_roc_from_dicts(fprs_list, tprs_list, classes):
    """
    Recebe listas por fold de dicionários: fprs_list[fold][classe]=array, idem tprs_list.
    Retorna dict por classe -> (mean_fpr, mean_tpr, auc_val) computado somente nos folds válidos.
    """
    mean_curves = {}
    base_fpr = np.linspace(0, 1, 200)
    for cls in classes:
        tprs_acc = []
        for fpr_d, tpr_d in zip(fprs_list, tprs_list):
            fpr = fpr_d.get(cls) if isinstance(fpr_d, dict) else None
            tpr = tpr_d.get(cls) if isinstance(tpr_d, dict) else None
            if fpr is None or tpr is None:
                continue
            tpr_i = np.interp(base_fpr, fpr, tpr)
            tpr_i[0] = 0.0
            tprs_acc.append(tpr_i)
        if len(tprs_acc) == 0:
            mean_curves[cls] = (None, None, None)
        else:
            mean_tpr = np.mean(tprs_acc, axis=0)
            mean_tpr[-1] = 1.0
            auc_val = auc(base_fpr, mean_tpr)
            mean_curves[cls] = (base_fpr, mean_tpr, auc_val)
    return mean_curves

def _compute_concat_roc_ovr(df_block, y_col, prob_cols):
    """Computa ROC OvR por classe usando dataframes concatenados (sem média entre folds).
       Retorna dict: classe -> (fpr, tpr, auc_val) e lista de classes.
    """
    classes = [int(c.split('_')[1]) for c in prob_cols]
    curves = {}
    y_true = df_block[y_col].values
    prob_mat = df_block[prob_cols].values
    for i, cls in enumerate(classes):
        y_bin = (y_true == cls).astype(int)
        pos = y_bin.sum()
        neg = len(y_bin) - pos
        if pos > 0 and neg > 0:
            fpr, tpr, _ = roc_curve(y_bin, prob_mat[:, i])
            curves[cls] = (fpr, tpr, auc(fpr, tpr))
        else:
            curves[cls] = (None, None, None)
    return curves, classes

# ---------- Leitura opcional de Val/Test (10%) para 8/1/1 ----------
val_df_811 = None
test10_df_811 = None
try:
    _val_csv = CSV_OUT / 'cv_811_validation.csv'
    if _val_csv.exists():
        tmp = pd.read_csv(_val_csv)
        tmp = tmp.copy()
        if 'y_val' in tmp.columns:
            tmp.rename(columns={'y_val':'y'}, inplace=True)
        val_df_811 = tmp
except Exception:
    pass

try:
    _test_csv = CSV_OUT / 'cv_811_test.csv'
    if _test_csv.exists():
        tmp = pd.read_csv(_test_csv)
        tmp = tmp.copy()
        if 'y_test' in tmp.columns:
            tmp.rename(columns={'y_test':'y'}, inplace=True)
        test10_df_811 = tmp
except Exception:
    pass

with PdfPages(PDF_OUT / f'RF_CV_SUMMARY_{N_SPLITS}.pdf') as pdf:
    # ------------------ Página de parâmetros ------------------
    fig, ax = plt.subplots(figsize=(10, 6))
    ax.axis('off')
    parametros = rf_params.copy()
    parametros['threshold'] = THRESHOLD
    parametros['n_splits'] = N_SPLITS
    header = 'Algoritmo: Random Forest\n'
    texto = header + '\n'.join([f'{k}: {v}' for k, v in parametros.items()])
    ax.text(0, 1, texto, fontsize=11, family='monospace', verticalalignment='top')
    ax.set_title('📌 Parâmetros do Modelo - Random Forest')
    try:
        png_path = IMAGES_OUT / f'params_{N_SPLITS}.png'
        fig.savefig(png_path, dpi=300, bbox_inches='tight')
    except Exception:
        pass
    pdf.savefig(fig); plt.close()

    # ------------------ Página de métricas resumidas ------------------
    if 'df_resultados' in globals() and isinstance(df_resultados, pd.DataFrame) and not df_resultados.empty:
        fig, ax = plt.subplots(figsize=(12, 6))
        ax.axis('off')
        display_df = df_resultados.groupby(['Conjunto']).mean(numeric_only=True).T.round(4)
        table = ax.table(cellText=display_df.values,
                         colLabels=display_df.columns,
                         rowLabels=display_df.index,
                         loc='center')
        table.auto_set_font_size(False)
        table.set_fontsize(9)
        try:
            png_path = IMAGES_OUT / f'metrics_summary_{N_SPLITS}.png'
            fig.savefig(png_path, dpi=300, bbox_inches='tight')
        except Exception:
            pass
        pdf.savefig(fig); plt.close()
    else:
        print("ℹ️ df_resultados não disponível ou vazio — página de métricas resumidas pulada.")

    # ------------------ Página ROC OvR por classe (Treino/Teste dos 80% internos) ------------------
    try:
        have_train_probs = ('train_df' in globals()
                            and isinstance(train_df, pd.DataFrame)
                            and not train_df.empty
                            and any(c.startswith('prob_') for c in train_df.columns)
                            and ('y_train' in train_df.columns))
        have_test_probs  = ('test_df' in globals()
                            and isinstance(test_df, pd.DataFrame)
                            and not test_df.empty
                            and any(c.startswith('prob_') for c in test_df.columns)
                            and ('y_test' in test_df.columns))

        curves_train = {}
        curves_test  = {}
        classes_all  = None

        if have_train_probs:
            classes_all, prob_cols_train = _get_classes_from_probs_columns(train_df)
            ctrain, _ = _compute_concat_roc_ovr(train_df, 'y_train', prob_cols_train)
            curves_train = ctrain
        if have_test_probs:
            classes_all_test, prob_cols_test = _get_classes_from_probs_columns(test_df)
            curves_test, _ = _compute_concat_roc_ovr(test_df, 'y_test', prob_cols_test)
            if classes_all is None:
                classes_all = classes_all_test

        # Fallback com médias por fold (caso não tenhamos concat)
        if (not have_train_probs) and ('fprs_train' in globals()) and len(fprs_train) > 0 and isinstance(fprs_train[0], dict):
            if classes_all is None:
                classes_all = sorted([int(k) for k in fprs_train[0].keys()])
            agg_train = _aggregate_mean_roc_from_dicts(fprs_train, tprs_train, classes_all)
            for cls, (mfpr, mtpr, mauc) in agg_train.items():
                curves_train[cls] = (mfpr, mtpr, mauc)

        if (not have_test_probs) and ('fprs_test' in globals()) and len(fprs_test) > 0 and isinstance(fprs_test[0], dict):
            if classes_all is None:
                classes_all = sorted([int(k) for k in fprs_test[0].keys()])
            agg_test = _aggregate_mean_roc_from_dicts(fprs_test, tprs_test, classes_all)
            for cls, (mfpr, mtpr, mauc) in agg_test.items():
                curves_test[cls] = (mfpr, mtpr, mauc)

        if classes_all is None:
            print('Não foi possível inferir classes para ROC OvR (train/test internos).')
        else:
            n_cls = len(classes_all)
            fig, axes = plt.subplots(1, n_cls, figsize=(5*n_cls, 5), squeeze=False)
            axes = axes[0]
            for j, cls in enumerate(classes_all):
                ax = axes[j]
                fpr_tr, tpr_tr, auc_tr = curves_train.get(cls, (None, None, None))
                fpr_ts, tpr_ts, auc_ts = curves_test.get(cls, (None, None, None))
                if fpr_tr is not None and tpr_tr is not None:
                    ax.plot(fpr_tr, tpr_tr, label=f'Treino(80%) AUC={auc_tr:.4f}', lw=2)
                else:
                    ax.plot([], [], label='Treino(80%) indisponível')
                if fpr_ts is not None and tpr_ts is not None:
                    ax.plot(fpr_ts, tpr_ts, label=f'TesteFold(80%) AUC={auc_ts:.4f}', lw=2)
                else:
                    ax.plot([], [], label='TesteFold(80%) indisponível')
                ax.plot([0, 1], [0, 1], 'k--', lw=1)
                ax.set_title(f'ROC OvR - Classe {cls}')
                ax.set_xlabel('FPR'); ax.set_ylabel('TPR')
                ax.legend(loc='lower right')
            plt.tight_layout()
            try:
                fig.savefig(IMAGES_OUT / f'roc_ovr_all_classes_{N_SPLITS}.png', dpi=300, bbox_inches='tight')
            except Exception:
                pass
            pdf.savefig(fig, bbox_inches='tight'); plt.close()
    except Exception as e:
        print('Não foi possível gerar ROC OvR (interno 80%):', e)

    # ------------------ Página ROC OvR (Validação 10% vs Teste 10%) — opcional 8/1/1 ------------------
    try:
        curves_val = {}
        curves_t10 = {}
        classes_all_10 = None

        # Val 10%
        if val_df_811 is not None and not val_df_811.empty:
            classes_all_10, prob_cols_val = _get_classes_from_probs_columns(val_df_811)
            cval, _ = _compute_concat_roc_ovr(val_df_811.rename(columns={'y':'y_val'}), 'y_val', prob_cols_val)
            curves_val = cval

        # Test 10%
        if test10_df_811 is not None and not test10_df_811.empty:
            classes_test_10, prob_cols_test10 = _get_classes_from_probs_columns(test10_df_811)
            ct10, _ = _compute_concat_roc_ovr(test10_df_811.rename(columns={'y':'y_test'}), 'y_test', prob_cols_test10)
            curves_t10 = ct10
            if classes_all_10 is None:
                classes_all_10 = classes_test_10

        if classes_all_10 is not None:
            n_cls = len(classes_all_10)
            fig, axes = plt.subplots(1, n_cls, figsize=(5*n_cls, 5), squeeze=False)
            axes = axes[0]
            for j, cls in enumerate(classes_all_10):
                ax = axes[j]
                fpr_v, tpr_v, auc_v = curves_val.get(cls, (None, None, None))
                fpr_t, tpr_t, auc_t = curves_t10.get(cls, (None, None, None))
                if fpr_v is not None and tpr_v is not None:
                    ax.plot(fpr_v, tpr_v, label=f'Val(10%) AUC={auc_v:.4f}', lw=2)
                else:
                    ax.plot([], [], label='Val(10%) indisponível')
                if fpr_t is not None and tpr_t is not None:
                    ax.plot(fpr_t, tpr_t, label=f'Test(10%) AUC={auc_t:.4f}', lw=2)
                else:
                    ax.plot([], [], label='Test(10%) indisponível')
                ax.plot([0, 1], [0, 1], 'k--', lw=1)
                ax.set_title(f'ROC OvR (10%) - Classe {cls}')
                ax.set_xlabel('FPR'); ax.set_ylabel('TPR')
                ax.legend(loc='lower right')
            plt.tight_layout()
            try:
                fig.savefig(IMAGES_OUT / f'roc_ovr_val_test10_{N_SPLITS}.png', dpi=300, bbox_inches='tight')
            except Exception:
                pass
            pdf.savefig(fig, bbox_inches='tight'); plt.close()
    except Exception as e:
        print('Não foi possível gerar ROC OvR (Val/Test 10%):', e)

    # ------------------ Página Confusion Matrix - Treino/Teste (internos 80%) ------------------
    try:
        def build_cm_from_df(df_block, true_col, pred_col, labels=None):
            y_true = df_block[true_col]
            y_pred = df_block[pred_col]
            if labels is None:
                labels = np.sort(pd.unique(np.concatenate([y_true.values, y_pred.values])))
            cm = confusion_matrix(y_true, y_pred, labels=labels)
            total = cm.sum()
            cm_percent = cm / total * 100 if total > 0 else np.zeros_like(cm, dtype=float)
            return cm, cm_percent, labels

        cm_train = cm_train_pct = labels_tr = None
        cm_test  = cm_test_pct  = labels_ts = None

        if ('train_df' in globals()
            and isinstance(train_df, pd.DataFrame)
            and not train_df.empty
            and ('y_train' in train_df.columns)
            and ('y_pred' in train_df.columns)):
            cm_train, cm_train_pct, labels_tr = build_cm_from_df(train_df, 'y_train', 'y_pred')
        if ('test_df' in globals()
            and isinstance(test_df, pd.DataFrame)
            and not test_df.empty
            and ('y_test' in test_df.columns)
            and ('y_pred' in test_df.columns)):
            cm_test, cm_test_pct, labels_ts = build_cm_from_df(test_df, 'y_test', 'y_pred')

        if cm_train is None and cm_test is None:
            print('Não há dados suficientes para gerar matrizes de confusão (interno 80%).')
        else:
            fig, axes = plt.subplots(1, 2, figsize=(12, 5))

            def plot_cm(ax, cm, cm_pct, labels, title):
                if cm is None:
                    ax.text(0.5, 0.5, 'Sem dados', ha='center', va='center'); ax.axis('off'); return
                annot = np.array([[f"{int(cm[i,j])}\n({cm_pct[i,j]:.2f}%)" for j in range(cm.shape[1])] for i in range(cm.shape[0])])
                sns.heatmap(cm, annot=annot, fmt='', cmap='Blues', ax=ax, cbar=False, annot_kws={'size':12})
                ax.set_xlabel('Predito'); ax.set_ylabel('Verdadeiro')
                ax.set_xticklabels([str(x) for x in labels], rotation=0)
                ax.set_yticklabels([str(x) for x in labels], rotation=0)
                ax.set_title(title)

            plot_cm(axes[0], cm_train, cm_train_pct, labels_tr, 'Matriz de Confusão - Treino(80%)')
            plot_cm(axes[1], cm_test,  cm_test_pct,  labels_ts, 'Matriz de Confusão - TesteFold(80%)')

            plt.tight_layout()
            try:
                fig.savefig(IMAGES_OUT / f'confusion_train_test_{N_SPLITS}.png', dpi=300, bbox_inches='tight')
            except Exception:
                pass
            pdf.savefig(fig, bbox_inches='tight'); plt.close()
    except Exception as e:
        print('Não foi possível gerar matrizes de confusão (interno 80%):', e)

print('PDF resumo gerado em:', PDF_OUT)


In [ ]:
# === Random Forest Básico (agora 9/1, mantendo compat c/ arquivos antigos e usando hold-out se disponível) ===
try:
    TEST_SIZE
except NameError:
    TEST_SIZE = 0.10  # 9/1 => 10% teste
try:
    RT_RANDOM_STATE
except NameError:
    RT_RANDOM_STATE = 42

from sklearn.ensemble import RandomForestClassifier
from sklearn.model_selection import train_test_split
from sklearn.metrics import accuracy_score, f1_score, precision_score, recall_score, roc_auc_score
from sklearn.preprocessing import OrdinalEncoder
import pandas as pd, numpy as np, joblib
from matplotlib.backends.backend_pdf import PdfPages
import matplotlib.pyplot as plt

# ---------------- Helpers (iguais/compatíveis aos seus) ----------------
def _apply_fixed_mapping_referral(df_in):
    if 'referral_source' in df_in.columns:
        mapped = df_in['referral_source'].map(VARIABLE_DISCRETIZE_NUMBER) if 'VARIABLE_DISCRETIZE_NUMBER' in globals() else df_in['referral_source']
        df_out = df_in.copy()
        df_out['referral_source'] = pd.Series(mapped).fillna(-1).astype('int64')
        return df_out
    return df_in

def _split_cat_num_columns(X, variable_discretize):
    non_numeric = [c for c in X.columns if not pd.api.types.is_numeric_dtype(X[c])]
    extra = [c for c in (variable_discretize if 'VARIABLE_DISCRETIZE' in globals() else [])
             if (c in X.columns and not pd.api.types.is_numeric_dtype(X[c]))]
    cat_cols = sorted(list(dict.fromkeys(non_numeric + extra)))
    num_cols = [c for c in X.columns if c not in cat_cols]
    return cat_cols, num_cols

def _fit_transform_train_test(Xtr, Xte):
    Xtr_m = _apply_fixed_mapping_referral(Xtr)
    Xte_m = _apply_fixed_mapping_referral(Xte)
    cat_cols, num_cols = _split_cat_num_columns(Xtr_m, VARIABLE_DISCRETIZE if 'VARIABLE_DISCRETIZE' in globals() else [])
    if len(cat_cols) > 0:
        enc = OrdinalEncoder(handle_unknown='use_encoded_value', unknown_value=-1, dtype=np.int64)
        Xtr_cat = enc.fit_transform(Xtr_m[cat_cols])
        Xte_cat = enc.transform(Xte_m[cat_cols])
        Xtr_enc = pd.concat(
            [
                Xtr_m[num_cols].reset_index(drop=True),
                pd.DataFrame(Xtr_cat, columns=cat_cols, index=Xtr_m.index).reset_index(drop=True),
            ],
            axis=1,
        )
        Xte_enc = pd.concat(
            [
                Xte_m[num_cols].reset_index(drop=True),
                pd.DataFrame(Xte_cat, columns=cat_cols, index=Xte_m.index).reset_index(drop=True),
            ],
            axis=1,
        )
    else:
        Xtr_enc, Xte_enc = Xtr_m.copy(), Xte_m.copy()
    # sanity
    if Xtr_enc.isna().any().any():
        raise ValueError(f"Há NaN em X_train codificado: {Xtr_enc.columns[Xtr_enc.isna().any()].tolist()}")
    if Xte_enc.isna().any().any():
        raise ValueError(f"Há NaN em X_test codificado: {Xte_enc.columns[Xte_enc.isna().any()].tolist()}")
    return Xtr_enc, Xte_enc

# ---------------- Pré-condições ----------------
if 'FEATURES' not in globals():
    raise RuntimeError('FEATURES não encontrado — execute as células anteriores para preparar os dados')
if 'df' not in globals():
    raise RuntimeError('Dataframe df não encontrado — execute as células anteriores para carregar os dados')

X_full = df[FEATURES].copy()
y_full = df['y'].copy()

# ===== Split 9/1 (estratificado) alinhado com nested 8/1/1 =====
# Se fold_indices (dos 10 folds) existir, usamos folds 1–9 como treino (90%)
# e fold 10 como teste (10%) = hold-out. Caso contrário, usamos train_test_split 9/1.
if 'fold_indices' in globals() and isinstance(fold_indices, (list, tuple)) and len(fold_indices) == 10:
    print("📐 [RF Básico] Usando folds 1–9 como treino (90%) e fold 10 como teste hold-out (10%) do esquema nested.")
    import numpy as _np
    train_1a9_idx = _np.concatenate(fold_indices[0:9])
    test_holdout_idx = fold_indices[9]

    # Índices originais do df são preservados
    X_train = X_full.iloc[train_1a9_idx].copy()
    X_test  = X_full.iloc[test_holdout_idx].copy()
    y_train = y_full.iloc[train_1a9_idx].copy()
    y_test  = y_full.iloc[test_holdout_idx].copy()
else:
    print("📐 [RF Básico] fold_indices não encontrado — usando train_test_split 9/1 padrão.")
    # 90% treino, 10% teste (fallback antigo)
    X_train_tmp, X_test_tmp, y_train_tmp, y_test_tmp = train_test_split(
        X_full.reset_index(), y_full.reset_index(),
        test_size=TEST_SIZE,
        stratify=y_full,
        random_state=RT_RANDOM_STATE
    )

    # Restaura índices originais como índice de linha
    def _restore(df_like):
        orig_idx = df_like['index'].values
        return df_like.drop(columns=['index']).set_index(pd.Index(orig_idx))

    X_train = _restore(X_train_tmp)
    X_test  = _restore(X_test_tmp)

    y_train = y_train_tmp.set_index(pd.Index(y_train_tmp['index'].values))['y']
    y_test  = y_test_tmp.set_index(pd.Index(y_test_tmp['index'].values))['y']

# ===== Encoding: fit no TREINO(90%), transform no TEST(10%) =====
X_train_enc, X_test_enc = _fit_transform_train_test(X_train, X_test)

# ===== Treino do modelo =====
classes_ordenadas = np.sort(pd.unique(y_full))
n_classes = len(classes_ordenadas)

model_basic = RandomForestClassifier(**rf_params)
model_basic.fit(X_train_enc, y_train)

# ===== Predições =====
pred_train = model_basic.predict(X_train_enc)
pred_test  = model_basic.predict(X_test_enc)

proba_train_all = model_basic.predict_proba(X_train_enc)
proba_test_all  = model_basic.predict_proba(X_test_enc)

if n_classes == 2:
    idx_pos = np.where(model_basic.classes_ == 1)[0][0]
    proba_train = proba_train_all[:, idx_pos]
    proba_test  = proba_test_all[:,  idx_pos]
else:
    # multi-classe: usamos probabilidade máxima apenas para logging simples;
    # para AUC usamos a matriz completa
    proba_train = proba_train_all.max(axis=1)
    proba_test  = proba_test_all.max(axis=1)

# ===== Métricas =====
def _metrics(y_true, y_pred, proba_all_or_vec, proba_all_full=None):
    if n_classes == 2:
        aucv = roc_auc_score(y_true, proba_all_or_vec) if len(np.unique(y_true)) > 1 else np.nan
        return dict(
            acc=accuracy_score(y_true, y_pred),
            f1=f1_score(y_true, y_pred),
            prec=precision_score(y_true, y_pred),
            rec=recall_score(y_true, y_pred),
            auc=aucv
        )
    else:
        # se proba_all_or_vec for matriz, usa direto; senão usa proba_all_full
        proba_for_auc = proba_all_or_vec if hasattr(proba_all_or_vec, "ndim") and proba_all_or_vec.ndim == 2 else proba_all_full
        try:
            aucv = roc_auc_score(y_true, proba_for_auc, multi_class='ovr', average='macro')
        except Exception:
            aucv = np.nan
        return dict(
            acc=accuracy_score(y_true, y_pred),
            f1=f1_score(y_true, y_pred, average='macro'),
            prec=precision_score(y_true, y_pred, average='macro', zero_division=0),
            rec=recall_score(y_true, y_pred, average='macro', zero_division=0),
            auc=aucv
        )

m_tr = _metrics(y_train, pred_train,
                proba_train if n_classes == 2 else proba_train_all,
                proba_all_full=proba_train_all)
m_ts = _metrics(y_test,  pred_test,
                proba_test if n_classes == 2 else proba_test_all,
                proba_all_full=proba_test_all)

print(f"[RF Básico 9/1] acc — train: {m_tr['acc']:.4f} | test: {m_ts['acc']:.4f}")

# ===== Salvamentos =====
# mantém padrão de nome com ts{TEST_SIZE*100}
ts_suffix = int(TEST_SIZE * 100)

model_name = MODEL_DIR / f'rf_basic_ts{ts_suffix}_rs{RT_RANDOM_STATE}.pkl'  # mantém o mesmo nome por compat
joblib.dump(model_basic, model_name)

# === SOMENTE DOIS ARQUIVOS X_ ENCODADOS (train/test) ===
X_train_enc.to_csv(BASICO_CSV / f'X_train_basic_ts{ts_suffix}_rs{RT_RANDOM_STATE}.csv')
X_test_enc.to_csv( BASICO_CSV / f'X_test_basic_ts{ts_suffix}_rs{RT_RANDOM_STATE}.csv')

# y_train / y_test (mantidos) — índices são os índices originais do df
pd.Series(y_train, name='y_train').to_csv(BASICO_CSV / f'y_train_basic_ts{ts_suffix}_rs{RT_RANDOM_STATE}.csv', index=True)
pd.Series(y_test,  name='y_test').to_csv( BASICO_CSV / f'y_test_basic_ts{ts_suffix}_rs{RT_RANDOM_STATE}.csv',  index=True)

# features após encoding
features = X_train_enc.columns.tolist()
joblib.dump(features, MODEL_DIR / f'features_basic_ts{ts_suffix}_rs{RT_RANDOM_STATE}.pkl')

# ===== CSV de métricas (mantém estrutura, val como NaN) =====
metrics_basic = pd.DataFrame([{
    'model': str(model_name.name),
    'split': '9/1',
    'random_state': RT_RANDOM_STATE,
    'acc_train': m_tr['acc'], 'f1_train': m_tr['f1'], 'precision_train': m_tr['prec'], 'recall_train': m_tr['rec'], 'auc_train': m_tr['auc'],
    'acc_val':   np.nan,       'f1_val':   np.nan,      'precision_val':   np.nan,      'recall_val':   np.nan,      'auc_val':   np.nan,
    'acc_test':  m_ts['acc'], 'f1_test':  m_ts['f1'], 'precision_test':  m_ts['prec'], 'recall_test':  m_ts['rec'], 'auc_test':  m_ts['auc']
}])
# mantém nome antigo do arquivo de métricas
metrics_basic.to_csv(BASICO_CSV / f'rf_model_basic.csv', index=False)

# ===== Dataset augmentado (mantém apenas TEST, como no original antes de adicionar VAL) =====
try:
    preds_test = pd.DataFrame(
        {
            'orig_index': X_test_enc.index,  # índice original do df
            'y_pred': pred_test,
            'y_proba': proba_test if n_classes == 2 else proba_test_all.max(axis=1),
        }
    )
    df_orig = df.reset_index().rename(columns={'index': 'orig_index'})
    df_aug_test = pd.merge(df_orig, preds_test, how='left', on='orig_index')
    df_aug_test.to_csv(
        BASICO_CSV / f'database_used_{DATASET_NAME}_with_preds_basic_ts{ts_suffix}_rs{RT_RANDOM_STATE}.csv',
        index=False,
    )
except Exception as e_aug:
    print('Não foi possível gerar dataset augmentado (TEST):', e_aug)

# ===== Amostras aleatórias do TEST para explicação (mantido) =====
num_instancias = 12
np.random.seed(42)
indices_aleatorios = np.random.choice(
    X_test_enc.index, size=min(num_instancias, len(X_test_enc)), replace=False
)

print('Modelo salvo em:', model_name)
print('Métricas salvas em:', BASICO_CSV / f'rf_model_basic.csv')
print('Dataset augmentado (TEST) salvo em:',
      BASICO_CSV / f'database_used_{DATASET_NAME}_with_preds_basic_ts{ts_suffix}_rs{RT_RANDOM_STATE}.csv')
print('Índices amostra aleatórios (TEST):', indices_aleatorios)


In [ ]:
# Salvar X_test_basic_full.csv e X_train_basic_full.csv (sem VAL — 9/1 ou qualquer split sem validação)
import pandas as pd
import numpy as np

# Carregar o arquivo cru
raw_df = pd.read_csv(DATASET_PATH)

def save_full_split(indices, y_true, y_pred, y_proba, split_name, classes=None):
    """
    indices  : índices do df original (devem bater com raw_df.index)
    y_true   : rótulos verdadeiros
    y_pred   : rótulos preditos
    y_proba  : vetor 1D (binário) OU matriz (n_amostras x n_classes)
    classes  : ordem das classes em y_proba (ex.: model_basic.classes_). Se None, assume 0..K-1
    """
    # garante que indices é um array/Index
    indices = np.asarray(indices)

    df_full = raw_df.loc[indices].copy()

    if isinstance(y_true, pd.Series):
        y_true = y_true.values
    if isinstance(y_pred, pd.Series):
        y_pred = y_pred.values

    df_full['y']      = y_true
    df_full['y_pred'] = y_pred

    y_proba = np.asarray(y_proba)
    if y_proba.ndim == 1:
        # binário: vetor 1D já é a prob da classe positiva
        df_full['y_proba'] = y_proba
    else:
        # multi-classe (ou probas completas, inclusive binário com 2 colunas)
        if classes is None:
            classes = list(range(y_proba.shape[1]))
        classes = [int(c) for c in classes]
        for j, cls in enumerate(classes):
            df_full[f'prob_{cls}'] = y_proba[:, j]
        prob_cols = [f'prob_{cls}' for cls in classes]
        df_full['y_proba_max']   = df_full[prob_cols].max(axis=1)
        df_full['y_pred_argmax'] = df_full[prob_cols].values.argmax(axis=1)

    df_full.to_csv(BASICO_CSV / f'X_{split_name}_basic_full.csv', index=True)
    print(f"✅ X_{split_name}_basic_full.csv salvo: {len(df_full)} linhas")


# ===== Índices (preservam índice original) =====
# Usar SEMPRE os índices de X_train/X_test, que carregam o índice original do df/raw_df.
idx_test_use  = X_test.index  if 'X_test'  in globals() else (X_test_enc.index  if 'X_test_enc'  in globals() else None)
idx_train_use = X_train.index if 'X_train' in globals() else (X_train_enc.index if 'X_train_enc' in globals() else None)

# ===== Compatibilidade de nomes das PREDIÇÕES (gera se não existir) =====
# Test
if 'y_pred_test_bin' not in globals():
    if 'pred_test' in globals():
        # para multiclasse, pred_test já é o rótulo final
        y_pred_test_bin = pred_test
    elif 'y_pred_test' in globals():
        y_pred_test_bin = y_pred_test
    else:
        y_pred_test_bin = model_basic.predict(X_test_enc if 'X_test_enc' in globals() else X_test)

# Train
if 'y_pred_train_bin' not in globals():
    if 'pred_train' in globals():
        y_pred_train_bin = pred_train
    elif 'y_pred_train' in globals():
        y_pred_train_bin = y_pred_train
    else:
        y_pred_train_bin = model_basic.predict(X_train_enc if 'X_train_enc' in globals() else X_train)

# ===== Probabilidades (usa já calculadas; senão recalcula) =====
try:
    proba_test_matrix  = proba_test_all
except NameError:
    proba_test_matrix  = model_basic.predict_proba(X_test_enc if 'X_test_enc' in globals() else X_test)

try:
    proba_train_matrix = proba_train_all
except NameError:
    proba_train_matrix = model_basic.predict_proba(X_train_enc if 'X_train_enc' in globals() else X_train)

# Ordem das classes
try:
    classes_order = model_basic.classes_
except Exception:
    classes_order = None

# ===== Salvamentos (APENAS TRAIN e TEST) =====
if idx_test_use is not None:
    save_full_split(
        idx_test_use,
        y_test,
        y_pred_test_bin,
        proba_test_matrix,
        'test',
        classes=classes_order
    )

if idx_train_use is not None:
    save_full_split(
        idx_train_use,
        y_train,
        y_pred_train_bin,
        proba_train_matrix,
        'train',
        classes=classes_order
    )

print("📁 Arquivos gerados em:", BASICO_CSV)
